# 02 — Literature & Scientific Evidence

**CuSelect-AI · Chapter II — From a falsifiable protocol to attributable evidence**  
**Campaign:** `CU-001` · **Scientific focus:** Cu-based catalysts and CO₂ reduction to ethylene · **Notebook status:** Evidence collection and verification workflow

**Previously established:** Notebook 00 verified the computational foundation and initial *unrelaxed* Cu/Cu–Ag geometries. Notebook 01 defined a falsifiable research question, outcome metrics, baselines and publication-level holdout policy. Neither notebook measured catalytic performance or validated a mechanistic claim.

**The question now:** *Which published claims can we identify, locate in original sources, distinguish by evidence type, and validate sufficiently to support later data-science and AI decisions?*

**Inputs**: `configs/CU-001_protocol.json`, `data/papers.csv` (eight **unverified seeds**), `src/cuselect/evidence.py` and source publications opened by a human reviewer.  
**Outputs**: bibliographic audit CSV, claim-review queue CSV, draft-claim CSV, an initially header-only verified-claims CSV, and a readiness report.  
**Acceptance gate**: No claim becomes *human_verified* merely because its DOI resolves, an abstract mentions it, or an LLM repeats it.

> **Research integrity:** This notebook does **not** download copyrighted PDFs automatically, predict ethylene FE, construct DFT reaction barriers, or claim the hypothesis has been demonstrated. Online metadata lookup is optional. Every scientific claim needs its precise source location and human verification before use as ground truth.

## 2.1 — Recover the approved research context

The literature search must start from the question we already defined, not from arbitrary interesting papers. Our protocol establishes **what counts as an error** and prevents us from changing evaluation criteria after inspecting results.

**Scientific boundary:** `CU-001-AG1` is a periodic, unrelaxed Cu(100) substitution model; the experimental paper on Ag-modified Cu **nanowires** is a different catalyst system. Its results may motivate a hypothesis, but they cannot automatically be assigned to our slab.

In [1]:
# PSEUDOCODE
# 1. Locate the project root regardless of VS Code's working directory.
# 2. Load the approved study protocol and eight seed bibliography records.
# 3. State the hypothesis and verify that evaluation is not already declared complete.

from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from cuselect.project_checks import require_project_root
from cuselect.protocol import load_research_protocol
from cuselect.evidence import (normalize_doi, audit_seed_bibliography,
    verify_metadata, EvidenceClaim, check_claim_links, claims_frame, save_csv)

start = Path.cwd().resolve()
for candidate_root in (start, *start.parents):
    if (candidate_root / 'pyproject.toml').is_file():
        ROOT = require_project_root(candidate_root)
        break
else:
    raise RuntimeError('Project root not found; open a notebook inside CuSelect-AI.')

protocol = load_research_protocol(ROOT / 'configs' / 'CU-001_protocol.json')
papers = pd.read_csv(ROOT / 'data' / 'papers.csv', dtype={'paper_id':'string','doi':'string'})
assert protocol.campaign_id == 'CU-001'
assert protocol.evaluation.evaluation_status == 'planned_not_run'
assert len(papers) == 8
print('Project:', ROOT.name)
print('Research campaign:', protocol.campaign_id)
print('Target:', protocol.system.target_product)
print('Seed bibliography:', len(papers), 'entries; not yet claim-verified')

Project: CuSelect-AI
Research campaign: CU-001
Target: ethylene (C2H4)
Seed bibliography: 8 entries; not yet claim-verified


## 2.2 — Bibliographic sanity checks: identifiers are not evidence

A **Digital Object Identifier (DOI)** identifies a registered scholarly object. Before reading a paper, check the DOI's syntax, duplicates, and expected publication year. Later we compare its title and year with publisher-deposited metadata.

**Important levels:** (i) valid-looking DOI, (ii) matching publication metadata, (iii) article and licensing reviewed, (iv) *specific scientific claim checked against a source location*. Only the fourth level can support a verified scientific claim.

In [2]:
# PSEUDOCODE
# 1. Normalize every DOI; verify only syntax and duplicates.
# 2. Display problems; preserve the original bibliographic seeds.
# 3. Fail early if the seed list itself contains invalid identifiers.

seed_audit = audit_seed_bibliography(papers)
display(seed_audit[['paper_id','doi','doi_syntax_valid','duplicate_doi','publication_year_valid','bibliographic_status']])
assert seed_audit['doi_syntax_valid'].all(), 'At least one seed DOI has invalid syntax.'
assert not seed_audit['duplicate_doi'].any(), 'Duplicate DOIs require review.'
assert not seed_audit['duplicate_paper_id'].any(), 'Duplicate internal paper IDs require review.'
assert seed_audit['publication_year_valid'].all(), 'Publication years require manual correction.'
print('DOI syntax and duplicate checks passed. Scientific claims are NOT verified.')

,paper_id,doi,doi_syntax_valid,duplicate_doi,publication_year_valid,bibliographic_status
0,P001,10.1002/anie.202524612,True,False,True,seed_not_verified
1,P002,10.1038/s41524-026-02139-1,True,False,True,seed_not_verified
2,P003,10.1021/acscentsci.6c01260,True,False,True,seed_not_verified
3,P004,10.1038/s44387-026-00111-4,True,False,True,seed_not_verified
4,P005,10.1038/s41467-024-54636-w,True,False,True,seed_not_verified
5,P006,10.1038/s41597-024-03180-9,True,False,True,seed_not_verified
6,P007,10.1002/anie.5598898,True,False,True,seed_not_verified
7,P008,10.1016/j.esci.2026.100642,True,False,True,seed_not_verified


DOI syntax and duplicate checks passed. Scientific claims are NOT verified.


## 2.3 — Optional publisher-deposited metadata retrieval

**Why Crossref?** Crossref provides a public REST API for scholarly metadata, including journal, title, DOI, and publication dates. Its metadata is deposited by publishers and other members. Metadata confirmation does **not** validate a paper's experiments, DFT calculations, or mechanisms.

- By default, this revised notebook runs **offline**. If a compatible prior Crossref metadata audit exists in `data/interim/02_metadata_audit.csv`, it can be reused and clearly identified as cached data. Otherwise records are labeled `not_queried`.
- When you have an internet connection on your Mac, change `RUN_LIVE_CROSSREF_LOOKUP` to `True`. Each request has a timeout; failures are recorded rather than silently treated as success.
- A title/year discrepancy triggers human review. Differences between publication-online and issue year can be legitimate.
- Copyright and licensing require a separate check at the article level; Crossref's presence of license metadata is not a full license review.

Official documentation: https://www.crossref.org/documentation/retrieve-metadata/rest-api/
- The cache is a record of a **previous lookup**, not evidence that Crossref was contacted in the current run. Neither a cache hit nor a fresh metadata match verifies scientific claims.


In [3]:
# PSEUDOCODE
# 1. Decide whether to query Crossref again or reuse a previously saved audit.
# 2. If using cached metadata, check that every paper ID and DOI still matches our seeds.
# 3. Never equate bibliographic metadata with scientific claim verification.

RUN_LIVE_CROSSREF_LOOKUP = False  # Set True to perform a NEW Crossref lookup on your Mac.
REUSE_PREVIOUS_METADATA_AUDIT = True
prior_audit_path = ROOT / 'data' / 'interim' / '02_metadata_audit.csv'

if RUN_LIVE_CROSSREF_LOOKUP:
    metadata_audit = verify_metadata(papers, online=True)
    metadata_provenance = 'live_crossref_lookup_this_run'
elif REUSE_PREVIOUS_METADATA_AUDIT and prior_audit_path.exists():
    previous = pd.read_csv(prior_audit_path, dtype={'paper_id':'string', 'doi':'string'})
    seed_pairs = {(str(x.paper_id), normalize_doi(str(x.doi))) for x in papers.itertuples(index=False)}
    prior_pairs = {(str(x.paper_id), normalize_doi(str(x.doi))) for x in previous.itertuples(index=False)}
    required = {'metadata_status','title_similarity','year_consistent','claim_review_status'}
    if seed_pairs != prior_pairs or not required.issubset(previous.columns) or len(previous) != len(papers):
        raise ValueError('Previous metadata audit does not match current bibliography; rerun Crossref lookup.')
    metadata_audit = previous.copy()
    metadata_provenance = ('previous_offline_audit_reused_not_queried'
                           if previous['metadata_status'].eq('not_queried').all()
                           else 'previous_metadata_audit_reused_not_requeried')
else:
    metadata_audit = verify_metadata(papers, online=False)
    metadata_provenance = 'not_queried'

metadata_cols = ['paper_id','doi','metadata_status','title_similarity','year_consistent','claim_review_status']
display(metadata_audit[metadata_cols])
print('Metadata provenance:', metadata_provenance)
print('Metadata review states:', metadata_audit['metadata_status'].value_counts().to_dict())
print('A DOI/title/year match does NOT verify a scientific claim.')

,paper_id,doi,metadata_status,title_similarity,year_consistent,claim_review_status
0,P001,10.1002/anie.202524612,matching_metadata_needs_manual_review,0.967,True,not_started
1,P002,10.1038/s41524-026-02139-1,matching_metadata_needs_manual_review,1.000,True,not_started
2,P003,10.1021/acscentsci.6c01260,matching_metadata_needs_manual_review,1.000,True,not_started
3,P004,10.1038/s44387-026-00111-4,matching_metadata_needs_manual_review,1.000,True,not_started
4,P005,10.1038/s41467-024-54636-w,matching_metadata_needs_manual_review,1.000,True,not_started
5,P006,10.1038/s41597-024-03180-9,matching_metadata_needs_manual_review,1.000,True,not_started
6,P007,10.1002/anie.5598898,matching_metadata_needs_manual_review,0.931,True,not_started
7,P008,10.1016/j.esci.2026.100642,matching_metadata_needs_manual_review,1.000,True,not_started


Metadata provenance: previous_metadata_audit_reused_not_requeried
Metadata review states: {'matching_metadata_needs_manual_review': 8}
A DOI/title/year match does NOT verify a scientific claim.


## 2.3b — A reproducible literature search, not a collection of convenient papers

The eight seeds are our *starting sample*, not a systematic review. Before expanding the evidence base, define the search objective, queries, databases, eligibility criteria, date and deduplication policy.

**Discovery sources:** Crossref and OpenAlex for broad bibliographic metadata; Semantic Scholar for scholarly search; publisher pages, institutional access, and discipline databases for full-text review. Scopus and Web of Science may require university subscriptions. Preprints must be labeled as such and not treated as equivalent to peer-reviewed articles.

**Inclusion priorities:** Cu or Cu-alloy CO₂RR studies with ethylene/ethanol mechanisms; agentic catalyst-discovery methods with clear baselines; applicable atomistic ML and interface models; authoritative catalyst datasets. **Exclude or separately label:** unrelated chemistries, unsupported claims, retracted papers, duplicate records and findings lacking extractable conditions.

The next cell records a **search plan only**. It does not secretly query external search engines. When searches are actually performed, record the search date, engine, result count, inclusions/exclusions and reasons, and append newly confirmed records to a reviewed bibliography—not directly to the evidence ledger.

In [4]:
# PSEUDOCODE
# 1. Predefine searches across chemistry, electrochemistry, materials and AI.
# 2. Record scope and review status before actually executing the searches.
# 3. Save a transparent plan; do not pretend planned searches have been run.

search_plan = pd.DataFrame([
    {'research_stream':'CO2RR Cu-Ag mechanisms','search_query':'("CO2 reduction" OR CO2RR) AND (Cu Ag OR copper silver) AND (ethylene OR ethanol) AND (mechanism OR coupling)'},
    {'research_stream':'Cu single-atom alloys','search_query':'copper single-atom alloy CO2 electroreduction C2+ selectivity'},
    {'research_stream':'AI catalyst workflows','search_query':'multi-agent LLM catalyst discovery DFT machine learning'},
    {'research_stream':'Electrode interface ML','search_query':'MLIP copper water interface constant potential electrochemistry'},
    {'research_stream':'Electrochemical datasets','search_query':'CO2 reduction catalyst dataset experimental conditions faradaic efficiency'},
    {'research_stream':'Scientific verification','search_query':'verification first autonomous catalysis evidence uncertainty LLM'},
])
search_plan['suggested_sources'] = 'Crossref;OpenAlex;Semantic Scholar;publisher databases'
search_plan['search_status'] = 'planned_not_executed'
search_plan['searched_on'] = ''
search_plan['hits_found'] = ''
search_plan['screening_notes'] = ''
display(search_plan[['research_stream','search_query','search_status']])
assert (search_plan['search_status']=='planned_not_executed').all()


,research_stream,search_query,search_status
0,CO2RR Cu-Ag mechanisms,"(""CO2 reduction"" OR CO2RR) AND (Cu Ag OR coppe...",planned_not_executed
1,Cu single-atom alloys,copper single-atom alloy CO2 electroreduction ...,planned_not_executed
2,AI catalyst workflows,multi-agent LLM catalyst discovery DFT machine...,planned_not_executed
3,Electrode interface ML,MLIP copper water interface constant potential...,planned_not_executed
4,Electrochemical datasets,CO2 reduction catalyst dataset experimental co...,planned_not_executed
5,Scientific verification,verification first autonomous catalysis eviden...,planned_not_executed


## 2.4 — Read the papers according to scientific function

Our eight publications are not eight independent answers to the same chemistry question. Some discuss **CO₂RR experiments**, some describe **autonomous-agent architectures**, one is a **dataset paper**, and one considers **constant-potential molecular modeling**.

| Paper | First question for the reviewer | Key distinction to preserve |
|:--|:--|:--|
| P001 | What exactly is the Cu-alloy agent optimizing? | C₂⁺ is not necessarily C₂H₄ alone |
| P002 | What was autonomously optimized and against which baseline? | Agent performance does not prove experimental selectivity |
| P003 | Which chemistry does MAESTRO actually study? | Its published case is **ORR**, not Cu CO₂RR |
| P004 | Is the contribution experimental or conceptual? | A **Perspective** outlining verification-first catalysis |
| P005 | How do Ag nanoparticles and Ag single atoms affect the product ratio? | Ag-modified **nanowires** are not the 3×3 Cu(100) slab |
| P006 | What data and label quality exist in the CO₂RR corpus? | Extracted corpus fields are not automatically ground truth |
| P007 | What does Al/Zn dual-doping improve, and under what conditions? | Reported **C₂⁺** FE must not be relabeled ethylene FE |
| P008 | How is electrochemical potential represented? | Generic energy/force MLIPs are not constant-potential models |

This matrix tells us *what to inspect*; it is not an expert-verification certificate. Begin with publisher abstracts, methods, figures, supplements, and accessible full text. For each claim, identify the exact section, figure, table, or supplementary reference.

In [5]:
# PSEUDOCODE
# 1. Prepare one specific verification task per seed publication.
# 2. Link the task to the original paper ID and DOI.
# 3. Never mark a task as completed merely because the title was retrieved.

review_questions = {
    'P001':'Separate C2+ design claims from ethylene-specific evidence; identify DFT and experimental support.',
    'P002':'Extract the task, simulation budget, baseline, and evidence supporting claimed savings.',
    'P003':'Confirm the ORR task and isolate transferable software methods from nontransferable chemistry.',
    'P004':'Record recommended verification practices; label these as perspective/methodology claims.',
    'P005':'Check Ag nanoparticles vs Ag single atoms, ethanol/ethylene ratio, reactor and conditions.',
    'P006':'Locate the annotated CO2RR dataset, available downloads, label provenance and licenses.',
    'P007':'Differentiate C2+ FE from ethylene FE; identify dual dopants and flow-cell conditions.',
    'P008':'Inspect potential-control method and Cu/water model transferability claims.',
}
review_queue = papers[['paper_id','doi','kind']].copy()
review_queue['doi'] = review_queue['doi'].map(normalize_doi)
review_queue['article_url'] = 'https://doi.org/' + review_queue['doi']
review_queue['first_review_question'] = review_queue['paper_id'].map(review_questions)
review_queue['full_text_access'] = 'not_assessed'
review_queue['article_license'] = 'not_assessed'
review_queue['source_review_status'] = 'pending_human_review'
assert review_queue['first_review_question'].notna().all()
display(review_queue[['paper_id','kind','first_review_question','source_review_status']])

,paper_id,kind,first_review_question,source_review_status
0,P001,AI_and_Cu_CO2RR,Separate C2+ design claims from ethylene-speci...,pending_human_review
1,P002,multi_agent_DFT,"Extract the task, simulation budget, baseline,...",pending_human_review
2,P003,multi_agent_MLFF,Confirm the ORR task and isolate transferable ...,pending_human_review
3,P004,scientific_verification,Record recommended verification practices; lab...,pending_human_review
4,P005,Cu_Ag_experimental,"Check Ag nanoparticles vs Ag single atoms, eth...",pending_human_review
5,P006,scientific_dataset,"Locate the annotated CO2RR dataset, available ...",pending_human_review
6,P007,dual_site_experiment,Differentiate C2+ FE from ethylene FE; identif...,pending_human_review
7,P008,constant_potential_MLIP,Inspect potential-control method and Cu/water ...,pending_human_review


## 2.5 — A scientific claim is not a bibliographic record

We now distinguish five frequent kinds of scientific statements:

| Claim class | Typical example | What would justify the claim? |
|---|---|---|
| `experimental_observation` | A product ratio changed between two catalyst preparations | Methods, original measured data, uncertainty and conditions |
| `computed_result` | A modeled intermediate has a particular adsorption energy | Exact structures, calculation reference, model, convergence and units |
| `author_interpretation` | A mechanistic pathway is *proposed* to explain a trend | The authors' discussion plus supporting or competing evidence |
| `hypothesis` | A new substitution pattern *might* favor ethylene | Marked as untested until an appropriate validation |
| `methodology` / `review_context` | A multi-agent workflow or verification policy was proposed | Accurate characterization of that paper's scope |

**Never confuse** ethylene FE with ethanol/ethylene ratios, C₂⁺ FE, C₂⁺ partial current density, or CO adsorption energy. Never transfer a result between different electrodes, reactors, electrolytes, or potentials without explicit applicability assessment.

A claim may use an `Abstract` location initially, but that gives **abstract-level support only**. More specific mechanistic or numerical assertions may require figures, methods, tables and supplements.

## 2.6 — Create a *draft* evidence ledger (not a verified dataset)

To practice rigorous extraction, the next cell prepares a handful of **paraphrased review candidates** based on published abstracts. They are clearly marked `pending_source_review` and are **not** ground truth. We deliberately leave the numerical FE column empty: the Ag–Cu ratios are *product ratios*, not Faradaic efficiencies. A human must independently inspect the original paper and document the exact source location before changing any draft to `human_verified`.

**Review workflow:** open paper → inspect passage/figure → record exact location and conditions → check claim against source → annotate limitations → separately approve or reject. We store the reusable schema and validation rules in `src/cuselect/evidence.py`.

In [6]:
# PSEUDOCODE
# 1. Create a small collection of literature-guided questions as DRAFT claims.
# 2. Validate the schema, DOI linkage, split group and data ranges.
# 3. Keep review_status pending; nothing is promoted by code execution.

claim_drafts = [
    EvidenceClaim(
        evidence_id='E-DRAFT-001', paper_id='P005', doi='10.1038/s41467-024-54636-w',
        claim_paraphrase='Wang et al. report ethanol-to-ethylene product ratios of 0.41 for AgCu NW (Ag nanoparticles) and 4.26 for Ag1Cu NW (single Ag atoms).',
        claim_type='experimental_observation', source_extent='abstract', exact_source_location='Abstract, sentence reporting ethanol-to-ethylene ratios (independent reviewer check pending)',
        catalyst_system='AgCu NW (Ag nanoparticles) and Ag1Cu NW (single Ag atoms) on Cu nanowires; NOT our Cu(100) slab',
        target_or_observation='ethanol/ethylene product ratio; NOT ethylene FE',
        attribution_note='Publisher abstract reports product ratios, NOT ethylene FE; verify exact source independently before promotion.'),
    EvidenceClaim(
        evidence_id='E-DRAFT-002', paper_id='P003', doi='10.1021/acscentsci.6c01260',
        claim_paraphrase='MAESTRO describes a multi-agent catalyst design workflow evaluated for oxygen reduction rather than Cu-based CO2 reduction.',
        claim_type='methodology', source_extent='abstract', exact_source_location='Abstract (confirm independently)',
        target_or_observation='agent workflow methodology, ORR chemistry',
        attribution_note='Transferrable architecture is not direct Cu CO2RR performance evidence.'),
    EvidenceClaim(
        evidence_id='E-DRAFT-003', paper_id='P004', doi='10.1038/s44387-026-00111-4',
        claim_paraphrase='A perspective advocates verification-first autonomous catalysis, emphasizing evidence records and uncertainty-aware validation.',
        claim_type='review_context', source_extent='abstract', exact_source_location='Abstract (confirm independently)',
        target_or_observation='methodological guidance, not validation of CuSelect-AI',
        attribution_note='Perspective paper; do not label as CuSelect-AI experimental proof.'),
    EvidenceClaim(
        evidence_id='E-DRAFT-004', paper_id='P007', doi='10.1002/anie.5598898',
        claim_paraphrase='A dual-dopant Cu alloy study reports enhanced C2+ formation under specified flow-cell conditions.',
        claim_type='experimental_observation', source_extent='abstract', exact_source_location='Abstract (confirm independently)',
        target_or_observation='C2+ products; do not infer ethylene-specific FE',
        attribution_note='Review product definitions and conditions in full text.'),
]
check_claim_links(claim_drafts, papers)
drafts_df = claims_frame(claim_drafts)
display(drafts_df[['evidence_id','paper_id','claim_type','claim_paraphrase','review_status']])
assert drafts_df['review_status'].eq('pending_source_review').all()
print('Draft review candidates:', len(drafts_df), '| Human-verified claims: 0')

,evidence_id,paper_id,claim_type,claim_paraphrase,review_status
0,E-DRAFT-001,P005,experimental_observation,Wang et al. report ethanol-to-ethylene product...,pending_source_review
1,E-DRAFT-002,P003,methodology,MAESTRO describes a multi-agent catalyst desig...,pending_source_review
2,E-DRAFT-003,P004,review_context,A perspective advocates verification-first aut...,pending_source_review
3,E-DRAFT-004,P007,experimental_observation,A dual-dopant Cu alloy study reports enhanced ...,pending_source_review


Draft review candidates: 4 | Human-verified claims: 0


## 2.7 — Quality control, applicability, and contradictions

**Validation is multidimensional:** source authenticity, exact source location, evidence class, units, experimental context, chemical applicability and publication-level data leakage. A difference in reported product selectivity is **not automatically a contradiction** when active sites, reactor setups, potentials or product definitions differ.

Before a claim can support an AI decision, ask:

1. **Provenance:** Where precisely was the result reported? Abstract, page, figure, table or supplement?
2. **Evidence:** Is the statement measured, computed, interpreted, hypothesized, or a methodological opinion?
3. **Conditions:** Which catalyst architecture, current density, potential reference, electrolyte, cell and normalization?
4. **Transferability:** Is this comparable with the Cu(100) slab? Why or why not?
5. **Leakage:** Was this publication assigned to development or withheld evaluation? Can our RAG agent access it?

For our first pilot, P005 can guide early **development**, but if its findings are in the agent's retrieval corpus, **P005 cannot also serve as an unseen holdout paper**. Split at DOI/publication level and enforce access controls later.

In [7]:
# PSEUDOCODE
# 1. Check that draft claims have valid paper links and remain unapproved.
# 2. Confirm distinct claim types and publication-level holdout keys.
# 3. Prepare an explicit set of scientific gaps to investigate.

assert drafts_df['evidence_id'].is_unique
assert drafts_df['split_group'].eq(drafts_df['doi']).all()
assert drafts_df['faradaic_efficiency_percent'].isna().all(), 'Do not encode product ratios as FE.'

scientific_gaps = pd.DataFrame([
    {'topic':'Cu-Ag applicability', 'missing_evidence':'Surface morphology and operating conditions that make nanowire results transferable to a Cu(100) slab', 'status':'open'},
    {'topic':'Mechanism discrimination', 'missing_evidence':'Competing ethylene/ethanol pathways under comparable potential and surface conditions', 'status':'open'},
    {'topic':'Model validity', 'missing_evidence':'Independent atomistic benchmark and uncertainty for selected Cu-Ag configurations', 'status':'open'},
    {'topic':'Agent novelty', 'missing_evidence':'Held-out tests showing reduced false advancements without excessive false rejection', 'status':'open'},
])
display(scientific_gaps)

,topic,missing_evidence,status
0,Cu-Ag applicability,Surface morphology and operating conditions th...,open
1,Mechanism discrimination,Competing ethylene/ethanol pathways under comp...,open
2,Model validity,Independent atomistic benchmark and uncertaint...,open
3,Agent novelty,Held-out tests showing reduced false advanceme...,open


## 2.7a — P005 guided source review: a real experimental case, not an automatic verification

**Primary source:** Wang et al., *Nature Communications* **15**, 10247 (2024), DOI [10.1038/s41467-024-54636-w](https://doi.org/10.1038/s41467-024-54636-w).

**Source-grounded candidate finding (Abstract):** the authors report ethanol-to-ethylene product ratios of **0.41 for AgCu NW** (Ag nanoparticles decorating Cu nanowires) and **4.26 for Ag₁Cu NW** (single Ag atoms on Cu nanowires). **These numbers are product ratios, not Faradaic efficiencies.** The ratio's exact experimental conditions cannot be inferred from the abstract alone; do not fill them from unrelated measurements.

The full paper separately describes flow-cell maxima of **54.9% ethylene FE** for AgCu NW at **−1.1 V vs RHE**, and **56.3% ethanol FE** for Ag₁Cu NW at **−1.0 V vs RHE** (1 M KOH, as reported). These are **different observations** with different target products and must receive different evidence IDs if entered. The reported potentials were **not iR-corrected**. The quoted current densities are not to be relabeled as ethylene/ethanol *partial* current densities without checking the definition in the paper. Review **Figure 2, its caption, Results, and the cited Supplementary Figures** before extracting data. [Original publisher article](https://www.nature.com/articles/s41467-024-54636-w).

**Mechanistic distinction:** the authors propose different coupling routes and support them with operando spectroscopy, isotope experiments and DFT. Their mechanistic interpretation is **not** a direct experimental measurement of every elementary step. If entered, classify the proposed route as `author_interpretation` rather than `experimental_observation`.

**Model applicability:** none of these nanowire FE values is a prediction for our idealized, unrelaxed, periodically repeated Cu(100)/Ag slab. P005 is a **development example**, not an unseen benchmark test article.

**Reviewer checklist:** confirm the precise abstract sentence, identify each catalyst configuration, record the source location, review Fig. 2 and supplementary information, note conditions and limitations, and only then determine whether to explicitly mark `human_verified` in the separate review CSV. Code execution is **not** the human review.



### 2.7a.1 — Verification of Ethanol-to-Ethylene Product Ratios

**Source:** Wang et al. (2024), *Nature Communications*.
DOI: https://doi.org/10.1038/s41467-024-54636-w

**Objective:** Verify a published experimental observation concerning the influence of Ag configuration on the relative formation of ethanol and ethylene during electrochemical CO₂ reduction.

The authors report the following ethanol-to-ethylene product ratios:

| Catalyst | Ag configuration | Ethanol/ethylene ratio |
|---|---|---:|
| AgCu NW | Ag nanoparticles on Cu nanowires | 0.41 |
| Ag₁Cu NW | Isolated Ag atoms on Cu nanowires | 4.26 |

**Scientific interpretation:** The reported product ratios demonstrate that Ag configuration affects the relative production of ethanol and ethylene. These ratios must not be confused with Faradaic efficiencies or directly transferred to our idealized Cu(100) slab models.

**Source verification:** The values, catalyst identities, and comparison were independently checked against the original publisher abstract on 6 November 2026.

**Evidence management:**
- Evidence ID: `E-DRAFT-001`
- Source location: Abstract
- Review status: `human_verified`
- Human source check: `True`
- Approved evidence export: `data/processed/02_verified_claims.csv`

**Outcome:** One experimentally reported observation has been human-verified and exported to the processed evidence register. This does not establish a causal mechanism or validate any predictive model.


In [8]:
# PSEUDOCODE
# 1. Open the existing human-review file from its *correct* location.
# 2. Inspect the P005 draft and any existing human-review entry.
# 3. Keep all statuses unchanged: inspecting a paper does not verify it.

review_path = ROOT / 'data' / 'references' / '02_human_reviewed_claims.csv'
expected_columns = list(EvidenceClaim.model_fields)

if review_path.exists():
    review_table = pd.read_csv(review_path, dtype=str, keep_default_na=False)
    missing_cols = sorted(set(expected_columns) - set(review_table.columns))
    extra_cols = sorted(set(review_table.columns) - set(expected_columns))
    if missing_cols or extra_cols:
        raise ValueError(f'Human-review schema mismatch. Missing: {missing_cols}; extra: {extra_cols}')
else:
    review_table = pd.DataFrame(columns=expected_columns)
    print('Review file not found yet. Create it from the header template before human approval.')

p005_draft = drafts_df.loc[drafts_df['paper_id'].eq('P005')].copy()
p005_existing = review_table.loc[review_table['paper_id'].eq('P005')].copy()

print('P005 draft evidence records:', len(p005_draft))
print('P005 human-review file records:', len(p005_existing))
print('Review file:', review_path.relative_to(ROOT))
display(p005_draft[['evidence_id','claim_paraphrase','claim_type','review_status']])
if not p005_existing.empty:
    display(p005_existing[['evidence_id','exact_source_location','source_checked_by_human','review_status','reviewer_note']])

assert drafts_df['review_status'].eq('pending_source_review').all()
print('No scientific claims were promoted by this cell.')

P005 draft evidence records: 1
P005 human-review file records: 3
Review file: data/references/02_human_reviewed_claims.csv


,evidence_id,claim_paraphrase,claim_type,review_status
0,E-DRAFT-001,Wang et al. report ethanol-to-ethylene product...,experimental_observation,pending_source_review


,evidence_id,exact_source_location,source_checked_by_human,review_status,reviewer_note
0,E-DRAFT-001,"Wang et al. (2024), Abstract",True,human_verified,Human independently checked the original publi...
1,E-P005-FE-C2H4-001,CO2RR performance; Fig. 2c; Supplementary Figs...,True,human_verified,Human independently confirmed the published C2...
2,E-P005-FE-ETOH-001,CO2RR performance; Fig. 2c; Supplementary Figs...,True,human_verified,Human independently confirmed the published C2...


No scientific claims were promoted by this cell.


In [9]:

# PSEUDOCODE
# 1. Read the existing human-review register.
# 2. Locate the previously reviewed P005 product-ratio claim.
# 3. Confirm that the human-review decision is recorded.
# 4. Check the claim's source and reported values.
# 5. Do not create or change verification decisions.

import pandas as pd

review_path = (
    ROOT / "data/references/02_human_reviewed_claims.csv"
)

review_df = pd.read_csv(
    review_path,
    dtype=str,
    keep_default_na=False
)

record = review_df.loc[
    review_df["evidence_id"] == "E-DRAFT-001"
]

assert len(record) == 1, (
    "Expected exactly one P005 product-ratio record."
)

claim = record.iloc[0]

assert claim["paper_id"] == "P005"
assert claim["source_extent"] == "abstract"

assert claim["source_checked_by_human"].lower() == "true"
assert claim["review_status"] == "human_verified"

assert "0.41" in claim["claim_paraphrase"]
assert "4.26" in claim["claim_paraphrase"]

print("P005 product-ratio verification record: VALID")
print("Evidence ID:", claim["evidence_id"])
print("Review status:", claim["review_status"])
print("Human verification preserved: True")
print("No files modified.")


P005 product-ratio verification record: VALID
Evidence ID: E-DRAFT-001
Review status: human_verified
Human verification preserved: True
No files modified.


In [10]:

# PSEUDOCODE
# 1. Read the human-review register.
# 2. Read the processed verified-claims database.
# 3. Check whether our first verified record appears in both.
# 4. Report the result without changing any files.

import pandas as pd

review_path = ROOT / "data/references/02_human_reviewed_claims.csv"
verified_path = ROOT / "data/processed/02_verified_claims.csv"

review_df = pd.read_csv(review_path)
verified_df = pd.read_csv(verified_path)

claim_id = "E-DRAFT-001"

record = review_df.loc[
    review_df["evidence_id"] == claim_id
]

assert len(record) == 1, "Expected exactly one review record."

human_verified = (
    str(record.iloc[0]["source_checked_by_human"]).lower() == "true"
    and record.iloc[0]["review_status"] == "human_verified"
)

in_processed_database = (
    claim_id in verified_df["evidence_id"].values
)

print("Claim ID:", claim_id)
print("Human verification complete:", human_verified)
print("Records in human-review register:", len(review_df))
print("Records in processed database:", len(verified_df))
print("Claim included in processed database:", in_processed_database)

if human_verified and not in_processed_database:
    print("\nNext action: Regenerate the verified-evidence export.")
elif human_verified and in_processed_database:
    print("\nThe verified claim is present in both databases.")


Claim ID: E-DRAFT-001
Human verification complete: True
Records in human-review register: 3
Records in processed database: 3
Claim included in processed database: True

The verified claim is present in both databases.


In [11]:

# PSEUDOCODE
# 1. Load the current human-review register.
# 2. Load the current processed evidence database.
# 3. Count approved claims.
# 4. Check whether approved IDs are present in the export.
# 5. Do not modify or overwrite any files.

import pandas as pd

review_path = ROOT / "data/references/02_human_reviewed_claims.csv"
verified_path = ROOT / "data/processed/02_verified_claims.csv"

reviews = pd.read_csv(
    review_path, dtype=str, keep_default_na=False
)

verified = pd.read_csv(
    verified_path, dtype=str, keep_default_na=False
)

human_checked = (
    reviews["source_checked_by_human"]
    .str.strip()
    .str.lower()
    .eq("true")
)

approved = reviews.loc[
    human_checked
    & reviews["review_status"].eq("human_verified")
].copy()

assert reviews["evidence_id"].is_unique
assert verified["evidence_id"].is_unique

approved_ids = set(approved["evidence_id"])
exported_ids = set(verified["evidence_id"])

print("Human-verified claims:", len(approved))
print("Previously exported claims:", len(verified))
print(
    "All approved IDs present in export:",
    approved_ids.issubset(exported_ids)
)

print("\nThis cell is read-only.")
print("Canonical export is performed in Section 2.8.")
print("No files modified.")


Human-verified claims: 3
Previously exported claims: 3
All approved IDs present in export: True

This cell is read-only.
Canonical export is performed in Section 2.8.
No files modified.


### 2.7a.2 — Extraction of Faradaic-Efficiency Evidence from P005

This subsection extracts two experimental observations from Wang et al. (2024): the maximum ethylene Faradaic efficiency on AgCu nanowires and the maximum ethanol Faradaic efficiency on Ag₁Cu nanowires.

The observations are recorded separately from the previously verified ethanol-to-ethylene product ratios. Both records remain pending human verification until the reported values and experimental conditions are independently checked against the original publication.

In [12]:

# PSEUDOCODE
# 1. Load the human-review register.
# 2. Prepare two separate P005 FE observations.
# 3. Mark both as pending human review.
# 4. Avoid duplicate evidence IDs.
# 5. Save the updated review register.

import pandas as pd

review_path = ROOT / "data/references/02_human_reviewed_claims.csv"
reviews = pd.read_csv(review_path, keep_default_na=False)

doi = "10.1038/s41467-024-54636-w"

observations = [
    {
        "evidence_id": "E-P005-FE-C2H4-001",
        "paper_id": "P005",
        "doi": doi,
        "claim_paraphrase":
            "AgCu NW reached a maximum ethylene FE of 54.9% "
            "at -1.1 V vs RHE in a flow cell.",
        "claim_type": "experimental_observation",
        "source_extent": "full_text",
        "exact_source_location":
            "CO2RR performance; Fig. 2c; Supplementary "
            "Figs. 21b-23 and Table 5",
        "catalyst_system": "AgCu NW",
        "target_or_observation": "Ethylene FE",
        "conditions":
            "Flow cell; 1 M KOH; -1.1 V vs RHE; "
            "no iR correction",
        "faradaic_efficiency_percent": 54.9,
        "fe_product": "C2H4",
        "attribution_note":
            "Publisher-reported experimental maximum FE; "
            "not an ML prediction or H-cell result.",
        "source_checked_by_human": False,
        "review_status": "pending_source_review",
        "reviewer_note":
            "Confirm FE, potential, reactor and conditions "
            "against the original article.",
        "split_group": doi,
    },
    {
        "evidence_id": "E-P005-FE-ETOH-001",
        "paper_id": "P005",
        "doi": doi,
        "claim_paraphrase":
            "Ag1Cu NW reached a maximum ethanol FE of 56.3% "
            "at -1.0 V vs RHE in a flow cell.",
        "claim_type": "experimental_observation",
        "source_extent": "full_text",
        "exact_source_location":
            "CO2RR performance; Fig. 2c; Supplementary "
            "Figs. 20 and 21a",
        "catalyst_system": "Ag1Cu NW",
        "target_or_observation": "Ethanol FE",
        "conditions":
            "Flow cell; 1 M KOH; -1.0 V vs RHE; "
            "no iR correction",
        "faradaic_efficiency_percent": 56.3,
        "fe_product": "C2H5OH",
        "attribution_note":
            "Publisher-reported experimental maximum FE; "
            "not an ML prediction or H-cell result.",
        "source_checked_by_human": False,
        "review_status": "pending_source_review",
        "reviewer_note":
            "Confirm FE, potential, reactor and conditions "
            "against the original article.",
        "split_group": doi,
    }
]

new_records = pd.DataFrame(observations)

# Preserve the approved evidence schema.
assert set(new_records.columns) == set(reviews.columns), (
    "Evidence schema mismatch; inspect before saving."
)

new_records = new_records[reviews.columns]

# Never overwrite existing evidence.
existing_ids = set(reviews["evidence_id"])
new_records = new_records.loc[
    ~new_records["evidence_id"].isin(existing_ids)
]

if not new_records.empty:
    reviews = pd.concat(
        [reviews, new_records],
        ignore_index=True
    )
    reviews.to_csv(review_path, index=False)

print("Total review records:", len(reviews))
print(
    "Human-verified records:",
    (reviews["review_status"] == "human_verified").sum()
)
print(
    "Pending records:",
    (reviews["review_status"] == "pending_source_review").sum()
)

display(
    reviews[
        [
            "evidence_id",
            "fe_product",
            "faradaic_efficiency_percent",
            "review_status"
        ]
    ]
)


Total review records: 3
Human-verified records: 3
Pending records: 0


,evidence_id,fe_product,faradaic_efficiency_percent,review_status
0,E-DRAFT-001,,,human_verified
1,E-P005-FE-C2H4-001,C2H4,54.9,human_verified
2,E-P005-FE-ETOH-001,C2H5OH,56.3,human_verified



### 2.7a.3 — Experimental Conditions and Evidence Validation

**Objective:** Validate the two Faradaic-efficiency observations extracted from P005 against the original publication.

**Source:** Wang et al. (2024), *Nature Communications*.
DOI: https://doi.org/10.1038/s41467-024-54636-w

#### Reported electrochemical performance

| Catalyst | Main product | Maximum FE (%) | Potential (V vs RHE) |
|---|---|---:|---:|
| AgCu NW | Ethylene | 54.9 | -1.1 |
| Ag₁Cu NW | Ethanol | 56.3 | -1.0 |

**Experimental conditions:**
- Electrochemical configuration: gas-fed flow cell
- Electrolyte: 1 M KOH
- Potential reference: reversible hydrogen electrode (RHE)
- Voltage correction: no iR correction
- Measurement: results collected after one hour of continuous electrolysis at each potential
- Figure 2 error bars: standard deviations of three independent measurements

**Source locations:** Results → CO₂RR performance; Figure 2c; relevant supplementary figures.

**Scientific distinction:** These measurements must not be confused with the lower Faradaic efficiencies obtained in the H-type cell using 0.1 M KHCO₃. The experimental nanowires also differ from the idealized Cu(100) models used in CuSelect-AI.

**Review procedure:** Inspect the reported values, catalyst identities, electrochemical conditions, and original figure before approving the two evidence records.

**Current status:** Both FE records remain pending human source review.


In [13]:

# PSEUDOCODE
# 1. Load the authoritative human-review register.
# 2. Locate the two P005 Faradaic-efficiency records.
# 3. Confirm their evidence IDs are unique.
# 4. Verify catalyst identities, products, FE values and potentials.
# 5. Confirm the published source and human-review status.
# 6. Display the validated experimental observations.
# 7. Do not modify scientific data or verification decisions.

import pandas as pd
from IPython.display import display

# --------------------------------------------------
# STEP 1 — Load the human-review register
# --------------------------------------------------

review_path = (
    ROOT / "data/references/02_human_reviewed_claims.csv"
)

reviews = pd.read_csv(
    review_path,
    dtype=str,
    keep_default_na=False
)

# --------------------------------------------------
# STEP 2 — Identify the two P005 FE records
# --------------------------------------------------

expected = {
    "E-P005-FE-C2H4-001": {
        "catalyst": "AgCu NW",
        "product": "C2H4",
        "fe": 54.9,
        "potential": "-1.1 V vs RHE"
    },
    "E-P005-FE-ETOH-001": {
        "catalyst": "Ag1Cu NW",
        "product": "C2H5OH",
        "fe": 56.3,
        "potential": "-1.0 V vs RHE"
    }
}

fe_records = reviews.loc[
    reviews["evidence_id"].isin(expected.keys())
].copy()

assert len(fe_records) == 2, (
    "Expected exactly two P005 FE records."
)

assert reviews["evidence_id"].is_unique, (
    "Duplicate evidence IDs detected."
)

# --------------------------------------------------
# STEP 3 — Validate scientific observations
# --------------------------------------------------

doi = "10.1038/s41467-024-54636-w"

for evidence_id, details in expected.items():

    matching = fe_records.loc[
        fe_records["evidence_id"] == evidence_id
    ]

    assert len(matching) == 1, (
        f"Missing or duplicate record: {evidence_id}"
    )

    claim = matching.iloc[0]

    # Publication and catalyst identity
    assert claim["paper_id"] == "P005"
    assert claim["doi"] == doi

    assert (
        claim["catalyst_system"] == details["catalyst"]
    )

    assert claim["fe_product"] == details["product"]

    # Reported Faradaic efficiency
    assert abs(
        float(claim["faradaic_efficiency_percent"])
        - details["fe"]
    ) < 1e-9, (
        f"Incorrect FE value for {evidence_id}"
    )

    # Electrochemical operating conditions
    assert details["potential"] in claim["conditions"]

    assert "Flow cell" in claim["conditions"]
    assert "1 M KOH" in claim["conditions"]

    # Source provenance
    assert claim["source_extent"] == "full_text"
    assert claim["exact_source_location"].strip()

    # Human verification must already exist.
    assert (
        claim["source_checked_by_human"]
        .strip()
        .lower() == "true"
    ), (
        f"Human source check missing: {evidence_id}"
    )

    assert claim["review_status"] == "human_verified"
    assert claim["reviewer_note"].strip()

# --------------------------------------------------
# STEP 4 — Display the validated evidence
# --------------------------------------------------

print("P005 — EXPERIMENTAL FE EVIDENCE VALIDATION")
print("=" * 50)

print("FE records checked:", len(fe_records))
print("Catalyst identities: VALID")
print("Faradaic efficiencies: VALID")
print("Operating conditions: VALID")
print("Publication and DOI: VALID")
print("Human verification: CONFIRMED")

print("\nVALIDATED EVIDENCE RECORDS")

display(
    fe_records[
        [
            "evidence_id",
            "catalyst_system",
            "fe_product",
            "faradaic_efficiency_percent",
            "conditions",
            "source_extent",
            "review_status"
        ]
    ]
)

print("\nNo evidence files were modified.")
print("No human-review decisions were created or changed.")


P005 — EXPERIMENTAL FE EVIDENCE VALIDATION
FE records checked: 2
Catalyst identities: VALID
Faradaic efficiencies: VALID
Operating conditions: VALID
Publication and DOI: VALID
Human verification: CONFIRMED

VALIDATED EVIDENCE RECORDS


,evidence_id,catalyst_system,fe_product,faradaic_efficiency_percent,conditions,source_extent,review_status
1,E-P005-FE-C2H4-001,AgCu NW,C2H4,54.9,Flow cell; 1 M KOH; -1.1 V vs RHE; no iR corre...,full_text,human_verified
2,E-P005-FE-ETOH-001,Ag1Cu NW,C2H5OH,56.3,Flow cell; 1 M KOH; -1.0 V vs RHE; no iR corre...,full_text,human_verified



No evidence files were modified.
No human-review decisions were created or changed.


In [14]:

# PSEUDOCODE
# 1. Load the existing human-review register.
# 2. Locate the two previously reviewed FE observations.
# 3. Confirm their scientific values and conditions.
# 4. Confirm that human verification is recorded.
# 5. Do not change approvals or write any files.

import pandas as pd

review_path = ROOT / "data/references/02_human_reviewed_claims.csv"

reviews = pd.read_csv(
    review_path,
    dtype=str,
    keep_default_na=False
)

expected = {
    "E-P005-FE-C2H4-001": {
        "catalyst": "AgCu NW",
        "product": "C2H4",
        "fe": 54.9,
        "potential": "-1.1 V vs RHE"
    },
    "E-P005-FE-ETOH-001": {
        "catalyst": "Ag1Cu NW",
        "product": "C2H5OH",
        "fe": 56.3,
        "potential": "-1.0 V vs RHE"
    }
}

assert reviews["evidence_id"].is_unique

for evidence_id, details in expected.items():

    record = reviews.loc[
        reviews["evidence_id"] == evidence_id
    ]

    assert len(record) == 1, (
        f"Expected one record for {evidence_id}"
    )

    claim = record.iloc[0]

    assert claim["paper_id"] == "P005"
    assert claim["doi"] == "10.1038/s41467-024-54636-w"
    assert claim["catalyst_system"] == details["catalyst"]
    assert claim["fe_product"] == details["product"]

    assert abs(
        float(claim["faradaic_efficiency_percent"])
        - details["fe"]
    ) < 1e-9

    assert details["potential"] in claim["conditions"]
    assert claim["source_extent"] == "full_text"

    assert (
        claim["source_checked_by_human"]
        .strip().lower() == "true"
    )

    assert claim["review_status"] == "human_verified"
    assert claim["exact_source_location"].strip()
    assert claim["reviewer_note"].strip()

print("P005 FARADAIC-EFFICIENCY VERIFICATION")
print("-" * 45)
print("Reviewed FE records:", len(expected))
print("Scientific values and conditions: VALID")
print("Human verification decisions: PRESERVED")
print("No files modified.")

display(
    reviews.loc[
        reviews["evidence_id"].isin(expected),
        [
            "evidence_id",
            "catalyst_system",
            "fe_product",
            "faradaic_efficiency_percent",
            "source_extent",
            "review_status"
        ]
    ]
)


P005 FARADAIC-EFFICIENCY VERIFICATION
---------------------------------------------
Reviewed FE records: 2
Scientific values and conditions: VALID
Human verification decisions: PRESERVED
No files modified.


,evidence_id,catalyst_system,fe_product,faradaic_efficiency_percent,source_extent,review_status
1,E-P005-FE-C2H4-001,AgCu NW,C2H4,54.9,full_text,human_verified
2,E-P005-FE-ETOH-001,Ag1Cu NW,C2H5OH,56.3,full_text,human_verified



### 2.7a.4 — Verified-Evidence Integrity and Readiness

**Objective:** Confirm that the approved evidence database is consistent with the human-review register before using scientific observations in downstream AI workflows.

The verification procedure checks:

1. Evidence IDs are unique.
2. Every exported claim has undergone human source review.
3. Every exported claim has an approved review status.
4. The approved evidence matches the human-review register.
5. Experimental Faradaic efficiencies are physically valid.
6. Product ratios are not incorrectly represented as Faradaic efficiencies.

**Acceptance criterion:** All integrity checks must pass before the evidence records can be considered eligible for downstream retrieval.

**Scope limitation:** Passing these checks does not demonstrate that the full literature database is complete, that a predictive model is accurate, or that an AI-generated catalyst recommendation is experimentally valid.



**Final evidence integrity assessment:** The authoritative full-record synchronization audit is performed in Section 2.8.8.1, following the canonical evidence export. That audit compares all fields in the approved evidence database with the corresponding human-reviewed records and checks that no unverified claims have been included.

This separation ensures that the final evidence-integrity assessment is performed after the verified dataset has been generated, preventing premature checks against outdated exported files.



### 2.7a.5 — Mechanistic Evidence and C–C Coupling Pathways

**Objective:** Investigate how the spatial configuration of Ag on Cu nanowires influences the proposed C–C coupling pathways toward ethylene and ethanol.

**Source:** Wang et al. (2024), *Nature Communications*.
DOI: https://doi.org/10.1038/s41467-024-54636-w

#### Mechanistic hypotheses

**AgCu NW — Ethylene-favored pathway**

The authors propose that Ag nanoparticles increase the availability of adsorbed *CO, favoring symmetric *CO–*CO coupling through a *COCO intermediate.

Supporting evidence includes operando Raman spectroscopy, ATR-SEIRAS measurements, and theoretical calculations.

**Ag₁Cu NW — Ethanol-favored pathway**

The authors propose that isolated Ag atoms facilitate water dissociation, promoting *CO hydrogenation to *CHO and subsequent asymmetric *CO–*CHO coupling through *COCHO.

Supporting evidence includes operando spectroscopy, isotope-labeling experiments, and DFT calculations.

#### Evidence classification

Spectroscopic signals and isotope-labeling results are experimental evidence, but assigning signals to specific adsorbed intermediates requires interpretation.

DFT calculations provide model-dependent mechanistic support rather than direct experimental proof of an elementary reaction pathway.

#### Scientific limitations

- The proposed mechanisms are specific to the investigated Ag-modified Cu nanowire systems.
- Spectroscopy and flow-cell performance measurements were conducted under different experimental conditions.
- Mechanistic interpretations must not be treated as measured Faradaic efficiencies.
- Results cannot be transferred automatically to CuSelect-AI's idealized Cu(100) and Ag-substituted Cu(100) models.

**Review status:** Mechanistic claims require separate source review and evidence records before they can be approved for downstream AI reasoning.


In [15]:

# PSEUDOCODE
# 1. Identify the two catalyst systems.
# 2. Summarize the proposed coupling pathways.
# 3. Record the evidence types and source figures.
# 4. Keep both mechanistic claims pending review.
# 5. Display the comparison without changing files.

import pandas as pd

mechanisms = pd.DataFrame([
    {
        "catalyst": "AgCu NW",
        "favored_product": "C2H4",
        "proposed_coupling": "*CO + *CO",
        "proposed_intermediate": "*COCO",
        "evidence_types": "Raman; ATR-SEIRAS; DFT",
        "source_location": "Figures 4 and 5; Results",
        "review_status": "pending_source_review"
    },
    {
        "catalyst": "Ag1Cu NW",
        "favored_product": "C2H5OH",
        "proposed_coupling": "*CO + *CHO",
        "proposed_intermediate": "*COCHO",
        "evidence_types": "Raman; ATR-SEIRAS; isotope labeling; DFT",
        "source_location": "Figures 4 and 5; Results",
        "review_status": "pending_source_review"
    }
])

assert len(mechanisms) == 2
assert mechanisms["catalyst"].is_unique

display(mechanisms)

print(
    "\nMechanistic hypotheses organized. "
    "No files modified and no claims approved."
)


,catalyst,favored_product,proposed_coupling,proposed_intermediate,evidence_types,source_location,review_status
0,AgCu NW,C2H4,*CO + *CO,*COCO,Raman; ATR-SEIRAS; DFT,Figures 4 and 5; Results,pending_source_review
1,Ag1Cu NW,C2H5OH,*CO + *CHO,*COCHO,Raman; ATR-SEIRAS; isotope labeling; DFT,Figures 4 and 5; Results,pending_source_review



Mechanistic hypotheses organized. No files modified and no claims approved.



#### 2.7a.5.1 — Spectroscopic Evidence for Proposed Reaction Intermediates

**Objective:** Examine the experimental evidence supporting the proposed symmetric and asymmetric C–C coupling mechanisms.

**Source:** Wang et al. (2024), Figure 4 and Results section: "In-situ capture of reaction intermediates."

The authors report ATR-SEIRAS and operando Raman signals attributed to *COCO, *CHO, *COCHO and *OH species.

**Evidence interpretation:** The spectral bands are experimental observations, while their assignment to particular surface intermediates involves interpretation. Correlations between signal intensity and product selectivity do not independently establish causality.

**Experimental conditions:** The operando spectroscopy was conducted in CO₂-saturated 0.1 M KHCO₃, unlike the 1 M KOH flow-cell experiments used for the previously verified maximum Faradaic efficiencies.

**Review status:** Pending independent examination of Figure 4 and the accompanying text.


In [16]:

# PSEUDOCODE
# 1. Organize reported spectral features from Figure 4.
# 2. Preserve the experimental technique and catalyst.
# 3. Distinguish observed bands from assigned species.
# 4. Keep records pending review without modifying files.

spectral_evidence = pd.DataFrame([
    {
        "catalyst": "AgCu NW",
        "technique": "ATR-SEIRAS",
        "band_cm-1": 1561,
        "assigned_species": "*COCO",
        "figure": "4d, 4f"
    },
    {
        "catalyst": "Ag1Cu NW",
        "technique": "ATR-SEIRAS",
        "band_cm-1": 1244,
        "assigned_species": "*CHO",
        "figure": "4e"
    },
    {
        "catalyst": "Ag1Cu NW",
        "technique": "ATR-SEIRAS",
        "band_cm-1": 1083,
        "assigned_species": "*COCHO",
        "figure": "4e, 4f"
    },
    {
        "catalyst": "Ag1Cu NW",
        "technique": "Raman",
        "band_cm-1": 537,
        "assigned_species": "*OH",
        "figure": "4b, 4c"
    }
])

spectral_evidence["review_status"] = "pending_source_review"

display(spectral_evidence)

print("Spectral features organized:", len(spectral_evidence))
print("No files were modified.")


,catalyst,technique,band_cm-1,assigned_species,figure,review_status
0,AgCu NW,ATR-SEIRAS,1561,*COCO,"4d, 4f",pending_source_review
1,Ag1Cu NW,ATR-SEIRAS,1244,*CHO,4e,pending_source_review
2,Ag1Cu NW,ATR-SEIRAS,1083,*COCHO,"4e, 4f",pending_source_review
3,Ag1Cu NW,Raman,537,*OH,"4b, 4c",pending_source_review


Spectral features organized: 4
No files were modified.



#### Original Figure 4 — Spectroscopic Source Evidence

**Publication:** Wang et al. (2024), Nature Communications, 15, 10247.

**Figure:** Figure 4 — In-situ capture of reaction intermediates.

[View the original Figure 4](https://www.nature.com/articles/s41467-024-54636-w/figures/4)

**Verified reported spectral assignments:**

| Catalyst | Technique | Band (cm⁻¹) | Assigned intermediate |
|---|---|---:|---|
| AgCu NW | ATR-SEIRAS | 1561 | *COCO |
| Ag₁Cu NW | ATR-SEIRAS | ~1244 | *CHO |
| Ag₁Cu NW | ATR-SEIRAS | ~1083 | *COCHO |
| Ag₁Cu NW | Raman | ~537 | *OH |

**Scientific interpretation:**

The reported spectral features are consistent with different surface-intermediate populations on AgCu NW and Ag₁Cu NW.

The authors use these observations to support their proposed symmetric and asymmetric C–C coupling mechanisms.

**Important distinction:** Verification of a reported spectral band and its published assignment does not independently establish that the assigned intermediate is unique or that a particular reaction pathway is dominant.

**Source-review status:** Reported bands and assignments checked against the original publication on 6 November 2026. Mechanistic conclusions remain subject to separate evidence assessment.

**DOI:** https://doi.org/10.1038/s41467-024-54636-w



### 2.7a.5.2 — DFT Evidence and Reaction-Pathway Analysis

**Objective:** Evaluate the computational evidence supporting the proposed C–C coupling mechanisms on Ag-modified Cu catalysts.

**Source:** Wang et al. (2024), Nature Communications, Figure 5 and Computational Methods.

DOI: https://doi.org/10.1038/s41467-024-54636-w

#### Computational methodology

The authors employed spin-polarized DFT calculations using:

- Software: VASP 6.3.0
- Exchange-correlation functional: PBE-GGA
- Dispersion correction: DFT-D3
- Model surface: Cu(111), 4 × 4 supercell
- Ag configurations: Isolated Ag atoms and Ag13 clusters
- Plane-wave cutoff: 400 eV
- k-point mesh for structural relaxation: 3 × 3 × 1
- k-point mesh for electronic calculations: 5 × 5 × 1
- Free-energy framework: Computational Hydrogen Electrode (CHE)
- Transition-state calculations: Climbing-image nudged elastic band (CI-NEB)

#### Principal theoretical findings

**Water adsorption and dissociation**

The calculations indicate more favorable H2O adsorption and a lower water-dissociation barrier on the modeled isolated-Ag catalyst.

**Formation of *CHO**

The proposed HO-Ag1-Cu-Cu environment lowers the calculated free-energy requirement for *CO hydrogenation to *CHO.

**C–C coupling**

The authors interpret the calculated free-energy landscapes as supporting asymmetric *CO–*CHO coupling toward ethanol on isolated-Ag-modified Cu, while Ag nanoparticles favor higher *CO availability and symmetric *CO–*CO coupling toward ethylene.

#### Scientific limitations

1. DFT-derived energies depend on the chosen surface models, computational approximations, and reaction conditions.
2. Calculated Gibbs free-energy differences must not be confused with transition-state activation barriers.
3. The proposed pathways are mechanistic interpretations supported by theory and experiments, not direct measurements of every elementary reaction step.
4. The published Cu(111) computational models differ from CuSelect-AI's initial Cu(100) slabs.
5. No DFT energy or activation barrier has yet been computed by CuSelect-AI.

**Review status:** The computational claims remain pending independent examination of Figure 5 and the relevant Supplementary Information.


In [17]:

# PSEUDOCODE
# 1. Define the five panels of Figure 5.
# 2. Identify the calculation or interpretation in each.
# 3. Classify the evidence correctly.
# 4. Record the published surface model.
# 5. Keep all claims pending source review.
# 6. Display results without changing files.

import pandas as pd

dft_evidence = pd.DataFrame([
    {
        "panel": "5a",
        "topic": "H2O adsorption",
        "evidence_type": "DFT adsorption energy",
        "finding": "Water adsorption is more favorable "
                   "on the modeled Ag1Cu surface.",
        "review_status": "pending_source_review"
    },
    {
        "panel": "5b",
        "topic": "H2O dissociation",
        "evidence_type": "DFT activation barrier",
        "finding": "The isolated-Ag site shows the "
                   "lowest calculated water-dissociation barrier.",
        "review_status": "pending_source_review"
    },
    {
        "panel": "5c",
        "topic": "*CO hydrogenation",
        "evidence_type": "DFT Gibbs free energy",
        "finding": "The proposed HO-Ag1Cu environment "
                   "favors formation of *CHO.",
        "review_status": "pending_source_review"
    },
    {
        "panel": "5d",
        "topic": "Product pathways",
        "evidence_type": "DFT Gibbs free-energy profile",
        "finding": "Computed pathways compare ethanol "
                   "and ethylene formation on Ag1Cu.",
        "review_status": "pending_source_review"
    },
    {
        "panel": "5e",
        "topic": "Proposed reaction pathway",
        "evidence_type": "Mechanistic interpretation",
        "finding": "The authors propose asymmetric "
                   "*CO-*CHO coupling toward ethanol.",
        "review_status": "pending_source_review"
    }
])

dft_evidence["doi"] = "10.1038/s41467-024-54636-w"
dft_evidence["published_model"] = "Cu(111) 4x4"
dft_evidence["our_initial_model"] = "Cu(100) 3x3"

assert len(dft_evidence) == 5
assert dft_evidence["panel"].is_unique
assert (
    dft_evidence["review_status"] == "pending_source_review"
).all()

display(dft_evidence)

print("DFT evidence topics organized:", len(dft_evidence))
print("Published model: Cu(111) 4x4")
print("CuSelect-AI model: Cu(100) 3x3")
print("No DFT calculations performed.")
print("No files modified or claims approved.")


,panel,topic,evidence_type,finding,review_status,doi,published_model,our_initial_model
0,5a,H2O adsorption,DFT adsorption energy,Water adsorption is more favorable on the mode...,pending_source_review,10.1038/s41467-024-54636-w,Cu(111) 4x4,Cu(100) 3x3
1,5b,H2O dissociation,DFT activation barrier,The isolated-Ag site shows the lowest calculat...,pending_source_review,10.1038/s41467-024-54636-w,Cu(111) 4x4,Cu(100) 3x3
2,5c,*CO hydrogenation,DFT Gibbs free energy,The proposed HO-Ag1Cu environment favors forma...,pending_source_review,10.1038/s41467-024-54636-w,Cu(111) 4x4,Cu(100) 3x3
3,5d,Product pathways,DFT Gibbs free-energy profile,Computed pathways compare ethanol and ethylene...,pending_source_review,10.1038/s41467-024-54636-w,Cu(111) 4x4,Cu(100) 3x3
4,5e,Proposed reaction pathway,Mechanistic interpretation,The authors propose asymmetric *CO-*CHO coupli...,pending_source_review,10.1038/s41467-024-54636-w,Cu(111) 4x4,Cu(100) 3x3


DFT evidence topics organized: 5
Published model: Cu(111) 4x4
CuSelect-AI model: Cu(100) 3x3
No DFT calculations performed.
No files modified or claims approved.



### 2.7a.5.3 — DFT Source Review and Model Comparability

**Objective:** Document the independent source review of Figure 5 and assess whether its computational findings can be transferred to CuSelect-AI's initial catalyst models.

**Source:** Wang et al. (2024), Nature Communications, Figure 5 and Computational Methods.

**Review date:** 6 November 2026.

**Source-review outcome:** All five Figure 5 panels were examined against the original publication. Their descriptions agree with the reported calculations and mechanistic interpretation.

**Computational-model limitation:** The paper employs Cu(111)-based models with isolated Ag atoms or Ag13 clusters. CuSelect-AI currently uses idealized Cu(100) and substitutional Ag/Cu(100) slabs. Direct quantitative transfer of energies or reaction barriers is not justified.

**Evidence classification:**
- DFT adsorption energies and activation barriers: published computational results.
- Gibbs free-energy diagrams: model-dependent thermodynamic calculations.
- Proposed C–C coupling pathways: mechanistic interpretations supported by experimental and computational evidence.

**Approval decision:** Figure descriptions are source-reviewed, but quantitative descriptors and mechanistic claims remain pending further assessment of the Supplementary Information and model applicability.

No DFT calculations have yet been performed by CuSelect-AI.


In [18]:

# PSEUDOCODE
# 1. Locate the previously saved Figure 5 review.
# 2. Load the existing human-review decisions.
# 3. Confirm that all five panels are represented.
# 4. Check that the figure review was previously completed.
# 5. Ensure mechanistic claims remain pending secondary review.
# 6. Display the results without modifying any files.

import pandas as pd
from IPython.display import display

# --------------------------------------------------
# STEP 1 — Locate the existing review file
# --------------------------------------------------

review_path = (
    ROOT / "data/interim/02_p005_dft_figure_review.csv"
)

assert review_path.exists(), (
    "Figure 5 review file not found. "
    "Recover the previously completed review record."
)

# --------------------------------------------------
# STEP 2 — Load previously recorded decisions
# --------------------------------------------------

dft_review = pd.read_csv(
    review_path,
    dtype=str,
    keep_default_na=False
)

# --------------------------------------------------
# STEP 3 — Validate the five figure panels
# --------------------------------------------------

expected_panels = {"5a", "5b", "5c", "5d", "5e"}

assert len(dft_review) == 5, (
    "Expected exactly five Figure 5 review records."
)

assert dft_review["panel"].is_unique, (
    "Duplicate Figure 5 panels detected."
)

assert set(dft_review["panel"]) == expected_panels, (
    "Figure 5 review contains missing or unexpected panels."
)

# --------------------------------------------------
# STEP 4 — Check previous human verification
# --------------------------------------------------

human_checks = (
    dft_review["figure_checked_by_human"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

assert human_checks.all(), (
    "One or more Figure 5 panels lack a recorded "
    "human source review."
)

# --------------------------------------------------
# STEP 5 — Preserve scientific review limitations
# --------------------------------------------------

assert dft_review["review_status"].eq(
    "pending_secondary_review"
).all(), (
    "Unexpected mechanistic review status."
)

# Published Cu(111) results must not be treated
# as directly comparable with our initial Cu(100) slabs.

assert dft_review["model_comparable_to_cuselect"].astype(
    str
).str.strip().str.lower().eq("false").all(), (
    "Unexpected model-comparability assessment."
)

# --------------------------------------------------
# STEP 6 — Display the review results
# --------------------------------------------------

print("P005 — FIGURE 5 SOURCE-REVIEW VALIDATION")
print("=" * 50)

print("Figure 5 panels:", len(dft_review))
print("Previously recorded human checks:", human_checks.sum())

print(
    "Approved mechanistic claims:",
    (dft_review["review_status"] == "human_verified").sum()
)

print("Model comparability: NOT ESTABLISHED")

display(
    dft_review[
        [
            "panel",
            "topic",
            "evidence_type",
            "figure_checked_by_human",
            "review_status"
        ]
    ]
)

print("\nNo files modified.")
print("No human-review decisions were created or changed.")


P005 — FIGURE 5 SOURCE-REVIEW VALIDATION
Figure 5 panels: 5
Previously recorded human checks: 5
Approved mechanistic claims: 0
Model comparability: NOT ESTABLISHED


,panel,topic,evidence_type,figure_checked_by_human,review_status
0,5a,H2O adsorption,DFT adsorption energy,True,pending_secondary_review
1,5b,H2O dissociation,DFT activation barrier,True,pending_secondary_review
2,5c,*CO hydrogenation,DFT Gibbs free energy,True,pending_secondary_review
3,5d,Product pathways,DFT Gibbs free-energy profile,True,pending_secondary_review
4,5e,Proposed reaction pathway,Mechanistic interpretation,True,pending_secondary_review



No files modified.
No human-review decisions were created or changed.



### 2.7a.6 — Numerical DFT Data Acquisition and Audit

**Objective:** Identify which numerical computational results from P005 are available in the authors' original source-data files.

**Publication:** Wang et al. (2024), Nature Communications.

**DOI:** https://doi.org/10.1038/s41467-024-54636-w

**Source-data repository:** https://doi.org/10.6084/m9.figshare.27199515

#### Data acquisition protocol

1. Obtain the original publisher-supplied Source Data file.
2. Preserve the original file without modification.
3. Inspect its worksheets and data organization.
4. Identify any numerical results corresponding to Figure 5.
5. Record units, catalyst models, calculation types, and source locations.
6. Distinguish published DFT data from measurements and mechanistic interpretations.

#### Scientific acceptance criteria

Numerical descriptors will be considered for modeling only when their meaning, units, calculation conditions, and provenance can be established.

Published Cu(111) calculations will not be treated as directly equivalent to CuSelect-AI's initial Cu(100) models.

**Current status:** Source-data inspection pending. No numerical DFT descriptors have been approved.


In [19]:

# PSEUDOCODE
# 1. Locate the original P005 source-data workbook.
# 2. Verify that the file exists.
# 3. Identify all available worksheets.
# 4. Display worksheet dimensions.
# 5. Do not change any data.

from pathlib import Path
import pandas as pd

source_path = ROOT / "data/raw/P005_source_data.xlsx"

assert source_path.exists(), (
    "Source-data file not found. "
    "Download the publisher XLSX and save it in data/raw/."
)

excel_file = pd.ExcelFile(source_path)

print("Source workbook:", source_path.name)
print("Number of worksheets:", len(excel_file.sheet_names))
print("\nWorksheet inventory:")

inventory = []

for sheet_name in excel_file.sheet_names:
    sheet = pd.read_excel(
        excel_file,
        sheet_name=sheet_name,
        header=None
    )

    inventory.append({
        "sheet_name": sheet_name,
        "rows": sheet.shape[0],
        "columns": sheet.shape[1]
    })

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

print("\nSource-data audit completed.")
print("No files were modified.")


Source workbook: P005_source_data.xlsx
Number of worksheets: 21

Worksheet inventory:


,sheet_name,rows,columns
0,Figure1f,102,59
1,Figure 1g,2702,10
2,Figure 1h,877,8
3,Figure 2a,1503,14
4,Figure 2b,9,43
5,Figure 2c,9,9
6,Figure 2d,25,8
7,Figure 3a,434,26
8,Figure 3b,134,21
9,Figure 3c,297,9



Source-data audit completed.
No files were modified.



#### 2.7a.6.1 — Inspection of Figure 5 Numerical Source Data

**Objective:** Examine the numerical source data associated with Figure 5a and Figure 5c of Wang et al. (2024).

**Source:** Publisher-supplied Excel workbook, `P005_source_data.xlsx`.

**Workbook audit:** The workbook contains 21 worksheets. Two worksheets correspond explicitly to Figure 5:

- **Figure 5a:** Water adsorption data.
- **Figure 5c:** Data associated with the Gibbs free-energy analysis of *CO hydrogenation.

Worksheets explicitly named Figure 5b, 5d and 5e are absent from this workbook.

**Procedure:** Inspect the raw cell values and their organization without modifying the original source file.

**Acceptance criteria:** Before accepting numerical descriptors, identify their units, catalyst configurations, calculation types, and original source locations.

**Current status:** Numerical content not yet inspected or approved for modeling.


In [20]:

# PSEUDOCODE
# 1. Select Figure 5a and Figure 5c.
# 2. Read their original cell values.
# 3. Display both datasets completely.
# 4. Do not modify the source workbook.
# 5. Do not approve numerical descriptors yet.

import pandas as pd

sheets_to_inspect = ["Figure 5a", "Figure 5c"]

assert all(
    name in excel_file.sheet_names
    for name in sheets_to_inspect
), "A required worksheet is missing."

for name in sheets_to_inspect:

    print(f"\n{'=' * 50}")
    print(f"WORKSHEET: {name}")
    print(f"{'=' * 50}")

    sheet = pd.read_excel(
        excel_file,
        sheet_name=name,
        header=None
    )

    print("Dimensions:", sheet.shape)
    print("Raw worksheet content:")

    display(sheet)

print("\nInspection complete.")
print("No source files were modified.")
print("No numerical DFT claims were approved.")



WORKSHEET: Figure 5a
Dimensions: (7, 4)
Raw worksheet content:


,0,1,2,3
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,H2O adsorption energies,NaN
4,NaN,NaN,Cu NW,-0.06094
5,NaN,NaN,Ag1Cu NW,-0.70452
6,NaN,NaN,AgCu NW,-0.04679



WORKSHEET: Figure 5c
Dimensions: (7, 6)
Raw worksheet content:


,0,1,2,3,4,5
0,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,CuNW,Ag1Cu NW,HO-Ag1Cu NW
5,NaN,NaN,*CO,0,0,--
6,NaN,NaN,*CHO,1.16,0.95,0.83



Inspection complete.
No source files were modified.
No numerical DFT claims were approved.



#### 2.7a.6.2 — Structured Extraction and Scientific Interpretation of Figure 5 Data

**Objective:** Convert the published numerical values from Figures 5a and 5c into structured, traceable computational evidence.

**Source:** Wang et al. (2024), publisher-provided Source Data workbook.

**Initial observations:**

- Figure 5a reports more negative H₂O adsorption energy for Ag₁Cu NW than for Cu NW or AgCu NW.
- Figure 5c reports *CHO free-energy values of 1.16, 0.95, and 0.83 for Cu NW, Ag₁Cu NW, and HO–Ag₁Cu NW, respectively.
- The *CO reference entry for HO–Ag₁Cu NW is absent in the source worksheet.

**Scientific cautions:**

1. Confirm the plotted energy units and reference states.
2. Do not confuse reaction Gibbs free energies with kinetic activation barriers.
3. Preserve missing values rather than replacing them with zero.
4. Distinguish published computational results from CuSelect-AI predictions.
5. Do not transfer Cu(111)-based numerical results directly to the Cu(100) models.

**Current status:** Source values extracted; numerical descriptors pending final metadata and comparability checks.


In [21]:

# PSEUDOCODE
# 1. Read the original Figure 5a and 5c worksheets.
# 2. Extract catalyst labels and published values.
# 3. Preserve missing reference values.
# 4. Record exact spreadsheet cell locations.
# 5. Display structured tables without saving or approving.

import pandas as pd

fig5a = pd.read_excel(
    source_path, sheet_name="Figure 5a", header=None
)

fig5c = pd.read_excel(
    source_path, sheet_name="Figure 5c", header=None
)

# Figure 5a: adsorption-energy values
adsorption = pd.DataFrame({
    "catalyst": fig5a.iloc[4:7, 2].tolist(),
    "reported_value": pd.to_numeric(
        fig5a.iloc[4:7, 3], errors="raise"
    ).tolist(),
    "source_sheet": "Figure 5a",
    "source_cell": ["D5", "D6", "D7"],
    "review_status": "units_pending_confirmation"
})

# Figure 5c: reported Gibbs free-energy levels
free_energy = pd.DataFrame({
    "catalyst": fig5c.iloc[4, 3:6].tolist(),
    "CO_reference": fig5c.iloc[5, 3:6].tolist(),
    "CHO_reported_value": pd.to_numeric(
        fig5c.iloc[6, 3:6], errors="raise"
    ).tolist(),
    "source_sheet": "Figure 5c",
    "source_cell": ["D7", "E7", "F7"],
    "review_status": "reference_pending_confirmation"
})

assert len(adsorption) == 3
assert len(free_energy) == 3
assert adsorption["reported_value"].notna().all()
assert free_energy["CHO_reported_value"].notna().all()

print("FIGURE 5a — WATER ADSORPTION")
display(adsorption)

print("\nFIGURE 5c — *CO / *CHO FREE-ENERGY DATA")
display(free_energy)

print("\nExtraction complete.")
print("Numerical descriptors approved: 0")
print("No files modified.")


FIGURE 5a — WATER ADSORPTION


,catalyst,reported_value,source_sheet,source_cell,review_status
0,Cu NW,-0.06094,Figure 5a,D5,units_pending_confirmation
1,Ag1Cu NW,-0.70452,Figure 5a,D6,units_pending_confirmation
2,AgCu NW,-0.04679,Figure 5a,D7,units_pending_confirmation



FIGURE 5c — *CO / *CHO FREE-ENERGY DATA


,catalyst,CO_reference,CHO_reported_value,source_sheet,source_cell,review_status
0,CuNW,0,1.16,Figure 5c,D7,reference_pending_confirmation
1,Ag1Cu NW,0,0.95,Figure 5c,E7,reference_pending_confirmation
2,HO-Ag1Cu NW,--,0.83,Figure 5c,F7,reference_pending_confirmation



Extraction complete.
Numerical descriptors approved: 0
No files modified.



#### 2.7a.6.3 — Energy Units and Reference-State Audit

**Objective:** Check the physical meaning and reference states of the published Figure 5 numerical values before accepting them as computational descriptors.

**Figure 5a:** H₂O adsorption energies, reported for Cu NW, Ag₁Cu NW and AgCu NW.

**Figure 5c:** Gibbs free-energy levels associated with *CO hydrogenation to *CHO on Cu NW, Ag₁Cu NW and HO–Ag₁Cu NW.

**Important distinctions:**

- Adsorption energies are not activation barriers.
- Reaction Gibbs free energies are not necessarily kinetic barriers.
- Missing source-data entries must remain missing.
- The HO–Ag₁Cu NW entry has no numerical *CO reference in the downloaded worksheet.
- Numerical comparisons require confirmed units, reference states and computational models.

**Status:** Numerical values extracted from the publisher workbook; final scientific metadata audit pending.


In [22]:

# PSEUDOCODE
# 1. Check the extracted Figure 5 values.
# 2. Convert numerical reference entries safely.
# 3. Preserve missing references as NaN.
# 4. Identify which comparisons have complete reference data.
# 5. Make no changes to source files.

import pandas as pd

energy_audit = free_energy.copy()

energy_audit["CO_reference_numeric"] = pd.to_numeric(
    energy_audit["CO_reference"],
    errors="coerce"
)

energy_audit["CHO_numeric"] = pd.to_numeric(
    energy_audit["CHO_reported_value"],
    errors="coerce"
)

energy_audit["CO_reference_available"] = (
    energy_audit["CO_reference_numeric"].notna()
)

energy_audit["CHO_minus_CO_if_defined"] = (
    energy_audit["CHO_numeric"]
    - energy_audit["CO_reference_numeric"]
)

energy_audit["unit_status"] = "pending_axis_confirmation"
energy_audit["reference_status"] = (
    energy_audit["CO_reference_available"]
    .map({
        True: "tabulated_reference_available",
        False: "reference_missing"
    })
)

display(
    energy_audit[
        [
            "catalyst",
            "CO_reference_numeric",
            "CHO_numeric",
            "CHO_minus_CO_if_defined",
            "unit_status",
            "reference_status"
        ]
    ]
)

print(
    "Missing CO references:",
    energy_audit["CO_reference_numeric"].isna().sum()
)

print("No files modified.")
print("No new descriptors approved.")


,catalyst,CO_reference_numeric,CHO_numeric,CHO_minus_CO_if_defined,unit_status,reference_status
0,CuNW,0.0,1.16,1.16,pending_axis_confirmation,tabulated_reference_available
1,Ag1Cu NW,0.0,0.95,0.95,pending_axis_confirmation,tabulated_reference_available
2,HO-Ag1Cu NW,NaN,0.83,NaN,pending_axis_confirmation,reference_missing


Missing CO references: 1
No files modified.
No new descriptors approved.



#### 2.7a.6.4 — Computational Descriptor Candidates and Provenance

**Objective:** Prepare structured numerical descriptor candidates from the publisher's Figure 5 source data.

The original figure confirms that both H₂O adsorption energies (Figure 5a) and *CHO free-energy levels (Figure 5c) are expressed in electronvolts (eV).

The original Excel-cell locations, catalyst labels, physical quantities, and review limitations are preserved.

**Important limitation:** The HO–Ag₁Cu NW *CO reference is missing in the source worksheet. Its reported *CHO value of 0.83 eV is retained without imputing a missing reference.

**Modeling status:** These values are published DFT results. They are not predictions generated by CuSelect-AI and are not yet approved as training descriptors for the initial Cu(100) models.


In [23]:

# PSEUDOCODE
# 1. Add confirmed units to the extracted source values.
# 2. Keep adsorption and free-energy quantities distinct.
# 3. Preserve missing source references.
# 4. Label all records as pending model-compatibility review.
# 5. Save an interim table for Notebook 03.

import pandas as pd

ads_candidates = adsorption.rename(
    columns={"reported_value": "value"}
).copy()

ads_candidates["quantity"] = "H2O_adsorption_energy"
ads_candidates["unit"] = "eV"
ads_candidates["reference_note"] = (
    "DFT adsorption-energy definition; Figure 5a"
)

cho_candidates = free_energy.rename(
    columns={"CHO_reported_value": "value"}
).copy()

cho_candidates["quantity"] = "CHO_free_energy_level"
cho_candidates["unit"] = "eV"

cho_candidates["reference_note"] = (
    cho_candidates["CO_reference"]
    .astype(str)
    .map(
        lambda x: (
            "CO reference tabulated as 0"
            if x == "0"
            else "CO reference not tabulated"
        )
    )
)

common_columns = [
    "catalyst",
    "quantity",
    "value",
    "unit",
    "source_sheet",
    "source_cell",
    "reference_note"
]

candidates = pd.concat(
    [
        ads_candidates[common_columns],
        cho_candidates[common_columns]
    ],
    ignore_index=True
)

candidates["doi"] = "10.1038/s41467-024-54636-w"
candidates["evidence_origin"] = "published_DFT"
candidates["model_compatibility"] = "not_established"
candidates["descriptor_approved"] = False

assert len(candidates) == 6
assert candidates["value"].notna().all()
assert candidates["unit"].eq("eV").all()

output_path = (
    ROOT / "data/interim/"
    "02_p005_computational_descriptor_candidates.csv"
)

candidates.to_csv(output_path, index=False)

display(candidates)

print("Descriptor candidates:", len(candidates))
print("Approved for modeling:", candidates["descriptor_approved"].sum())
print("Saved:", output_path.relative_to(ROOT))


,catalyst,quantity,value,unit,source_sheet,source_cell,reference_note,doi,evidence_origin,model_compatibility,descriptor_approved
0,Cu NW,H2O_adsorption_energy,-0.06094,eV,Figure 5a,D5,DFT adsorption-energy definition; Figure 5a,10.1038/s41467-024-54636-w,published_DFT,not_established,False
1,Ag1Cu NW,H2O_adsorption_energy,-0.70452,eV,Figure 5a,D6,DFT adsorption-energy definition; Figure 5a,10.1038/s41467-024-54636-w,published_DFT,not_established,False
2,AgCu NW,H2O_adsorption_energy,-0.04679,eV,Figure 5a,D7,DFT adsorption-energy definition; Figure 5a,10.1038/s41467-024-54636-w,published_DFT,not_established,False
3,CuNW,CHO_free_energy_level,1.16000,eV,Figure 5c,D7,CO reference tabulated as 0,10.1038/s41467-024-54636-w,published_DFT,not_established,False
4,Ag1Cu NW,CHO_free_energy_level,0.95000,eV,Figure 5c,E7,CO reference tabulated as 0,10.1038/s41467-024-54636-w,published_DFT,not_established,False
5,HO-Ag1Cu NW,CHO_free_energy_level,0.83000,eV,Figure 5c,F7,CO reference not tabulated,10.1038/s41467-024-54636-w,published_DFT,not_established,False


Descriptor candidates: 6
Approved for modeling: 0
Saved: data/interim/02_p005_computational_descriptor_candidates.csv



#### 2.7a.6.5 — Computational Data Quality Control

**Objective:** Confirm that all six computational descriptor candidates match their original published source-data cells.

**Quality-control criteria:**

1. Every numerical value must match its original Excel cell.
2. Each descriptor must retain its source worksheet and cell location.
3. Adsorption energies and Gibbs free-energy levels must remain distinct quantities.
4. Units and DOI metadata must be recorded.
5. Missing reference states must not be replaced with assumed values.
6. Published Cu(111)-based results must not be treated as directly transferable to CuSelect-AI's Cu(100) models.

**Dataset limitations:** The dataset contains only six values across two physical quantities and different catalyst configurations. It is insufficient for training a reliable predictive model.

**Acceptance decision:** Source-consistent values may be retained as literature-derived computational evidence. Approval for predictive modeling requires separate structural, thermodynamic, and data-quality assessments.


In [24]:

# PSEUDOCODE
# 1. Load the interim computational dataset.
# 2. Open the original publisher workbook in read-only mode.
# 3. Compare every saved numerical value with its Excel source cell.
# 4. Check provenance and scientific classifications.
# 5. Report the audit without changing any files.

import pandas as pd
from math import isclose
from openpyxl import load_workbook

dataset_path = (
    ROOT / "data/interim/"
    "02_p005_computational_descriptor_candidates.csv"
)

source_path = ROOT / "data/raw/P005_source_data.xlsx"

dataset = pd.read_csv(dataset_path)

workbook = load_workbook(
    source_path,
    read_only=True,
    data_only=True
)

audit_results = []

for _, row in dataset.iterrows():

    source_value = workbook[
        row["source_sheet"]
    ][row["source_cell"]].value

    values_match = isclose(
        float(row["value"]),
        float(source_value),
        rel_tol=0,
        abs_tol=1e-6
    )

    audit_results.append({
        "catalyst": row["catalyst"],
        "quantity": row["quantity"],
        "source": (
            f"{row['source_sheet']}!"
            f"{row['source_cell']}"
        ),
        "saved_value": row["value"],
        "original_value": source_value,
        "match": values_match
    })

workbook.close()

audit_df = pd.DataFrame(audit_results)

assert len(audit_df) == 6
assert audit_df["match"].all()
assert dataset["unit"].eq("eV").all()
assert dataset["doi"].eq(
    "10.1038/s41467-024-54636-w"
).all()

assert dataset["evidence_origin"].eq(
    "published_DFT"
).all()

approved = (
    dataset["descriptor_approved"]
    .astype(str)
    .str.lower()
    .eq("true")
)

assert not approved.any(), (
    "Some descriptors were unexpectedly approved."
)

display(audit_df)

print("Source-value matches:", audit_df["match"].sum())
print("Total numerical records:", len(dataset))
print("Approved modeling descriptors:", approved.sum())
print("Data-quality audit: PASSED")
print("No files modified.")


,catalyst,quantity,source,saved_value,original_value,match
0,Cu NW,H2O_adsorption_energy,Figure 5a!D5,-0.06094,-0.06094,True
1,Ag1Cu NW,H2O_adsorption_energy,Figure 5a!D6,-0.70452,-0.70452,True
2,AgCu NW,H2O_adsorption_energy,Figure 5a!D7,-0.04679,-0.04679,True
3,CuNW,CHO_free_energy_level,Figure 5c!D7,1.16000,1.16000,True
4,Ag1Cu NW,CHO_free_energy_level,Figure 5c!E7,0.95000,0.95000,True
5,HO-Ag1Cu NW,CHO_free_energy_level,Figure 5c!F7,0.83000,0.83000,True


Source-value matches: 6
Total numerical records: 6
Approved modeling descriptors: 0
Data-quality audit: PASSED
No files modified.



### 2.7a.7 — P005 Evidence Review Summary

**Publication:** Wang et al. (2024), Nature Communications.

**DOI:** https://doi.org/10.1038/s41467-024-54636-w

#### Completed activities

1. Verified the reported ethanol-to-ethylene product ratios.
2. Verified two experimental Faradaic-efficiency observations.
3. Examined the reported spectroscopic evidence and intermediate assignments.
4. Reviewed Figure 5 and its proposed reaction mechanisms.
5. Acquired the publisher's original source-data workbook.
6. Extracted six computational values from Figures 5a and 5c.
7. Confirmed that all six numerical values match their original Excel cells.

#### Evidence status

- Human-verified experimental claims: 3
- Computational descriptor candidates: 6
- Numerical descriptor transcription checks passed: 6
- Computational descriptors approved for modeling: 0
- Mechanistic claims approved: 0

#### Scientific limitations

The published computational models differ from CuSelect-AI's idealized Cu(100) structures. Quantitative transfer of energies and mechanistic conclusions requires additional justification.

The extracted six-value dataset is insufficient for training a reliable machine-learning model.

#### Conclusion

P005 provides an initial set of source-traceable experimental observations and published computational results.

The experimental observations are approved as literature evidence. Computational descriptors and mechanistic interpretations remain subject to further scientific and model-compatibility assessment.

**Next objective:** Evaluate the overall evidence-readiness status and continue reviewing the remaining seed publications.


In [25]:

# PSEUDOCODE
# 1. Load the bibliography and approved evidence.
# 2. Count publications represented by approved claims.
# 3. Count published computational descriptor candidates.
# 4. Identify publications without approved evidence.
# 5. Display the current research status without modifying files.

import pandas as pd

papers = pd.read_csv(ROOT / "data/papers.csv")

verified = pd.read_csv(
    ROOT / "data/processed/02_verified_claims.csv"
)

descriptors = pd.read_csv(
    ROOT / "data/interim/"
    "02_p005_computational_descriptor_candidates.csv"
)

covered_ids = set(verified["paper_id"])
all_ids = set(papers["paper_id"])

remaining_ids = sorted(all_ids - covered_ids)

print("CUSELECT-AI — LITERATURE EVIDENCE STATUS")
print("-" * 45)

print("Seed publications:", len(papers))
print("Publications with approved claims:", len(covered_ids))
print("Publications without approved claims:", len(remaining_ids))
print("Human-verified claims:", len(verified))
print("Computational descriptor candidates:", len(descriptors))

approved_descriptors = (
    descriptors["descriptor_approved"]
    .astype(str).str.lower().eq("true")
)

print("Descriptors approved for modeling:", approved_descriptors.sum())
print("\nPublications requiring further evidence review:")
print(", ".join(remaining_ids))

print("\nNo files were modified.")


CUSELECT-AI — LITERATURE EVIDENCE STATUS
---------------------------------------------
Seed publications: 8
Publications with approved claims: 1
Publications without approved claims: 7
Human-verified claims: 3
Computational descriptor candidates: 6
Descriptors approved for modeling: 0

Publications requiring further evidence review:
P001, P002, P003, P004, P006, P007, P008

No files were modified.



## 2.7b — Evidence Approval, Integrity, and Reproducibility

### Objective

Establish a reproducible and scientifically defensible procedure for approving published evidence before its use in CuSelect-AI's literature retrieval, multi-agent catalyst design, and scientific evaluation workflows.

The central principle is that **a scientific claim must not be considered verified merely because its DOI exists, its metadata are correct, or an AI system reports that it is supported.**

### Evidence Approval Criteria

A claim is eligible for inclusion in the verified evidence database only when:

1. **Source identification:** The original publication and DOI are correctly identified.
2. **Source localization:** The supporting abstract, section, figure, table, or supplementary source is recorded.
3. **Scientific accuracy:** Reported values, catalyst identities, and experimental or computational conditions match the original source.
4. **Evidence classification:** Experimental measurements, DFT calculations, mechanistic interpretations, and AI predictions are clearly distinguished.
5. **Human verification:** An independent reviewer has examined the source and confirmed the claim.
6. **Provenance integrity:** Evidence IDs, publication identifiers, source locations, and review decisions are preserved.

Approved claims must satisfy:

- `source_checked_by_human = True`
- `review_status = "human_verified"`

These fields must never be assigned automatically by an AI-generated summary or metadata lookup.

### Current Evidence Status

Following the initial review of Wang et al. (2024), P005:

| Evidence category | Number | Status |
|---|---:|---|
| Seed publications | 8 | Bibliographic register |
| Publications with approved claims | 1 | P005 |
| Human-verified experimental claims | 3 | Approved |
| Computational descriptor candidates | 6 | Extracted and source-value checked |
| Computational descriptors approved for modeling | 0 | Not approved |
| Mechanistic claims approved | 0 | Further assessment required |
| Publications without approved claims | 7 | Further review required |

The six computational values extracted from Figures 5a and 5c match their original Excel cells. However, numerical agreement does not establish model compatibility, computational reproducibility, or mechanistic validity.

### Evidence Storage

**Human-review register:**

`data/references/02_human_reviewed_claims.csv`

Stores individual claims, source details, human-review decisions, and reviewer notes.

**Verified evidence database:**

`data/processed/02_verified_claims.csv`

Contains claims that meet the approved source-review criteria.

**Computational descriptor candidates:**

`data/interim/02_p005_computational_descriptor_candidates.csv`

Contains published numerical DFT values and their provenance, without implying approval for machine-learning applications.

**Evidence-readiness report:**

`outputs/reports/02_evidence_readiness.json`

Records the state of the evidence workflow. This report must be updated from the current approved records rather than relying on outdated notebook outputs.

### Reproducibility and Data Integrity

The evidence pipeline must:

1. Preserve existing human-review decisions during repeated notebook execution.
2. Prevent unverified claims from entering the approved evidence database.
3. Detect duplicate evidence IDs and contradictory verification statuses.
4. Preserve units, missing values, experimental conditions, and computational reference information.
5. Maintain publication-level provenance to prevent train/test data leakage.
6. Regenerate derived reports without silently overwriting manually reviewed records.
7. Clearly distinguish literature-evidence readiness from predictive-model readiness.

### Scientific Limitations

Approval of a published observation confirms that it has been accurately represented from the reviewed source. It does not independently reproduce the underlying experiment or calculation.

The P005 Cu(111)-based theoretical models differ from CuSelect-AI's initial Cu(100) catalyst structures. Their numerical descriptors therefore require additional compatibility assessment before use in predictive modeling.

### Acceptance Decision

The three human-verified P005 experimental claims are eligible for evidence-based retrieval, subject to the system's provenance and consistency checks.

The six computational descriptor candidates and proposed mechanistic conclusions remain outside the approved modeling dataset.

**Next step:** Audit the current evidence-readiness report, ensure it reflects the three verified records, and verify that rerunning Notebook 02 cannot erase or falsely approve scientific evidence.


In [26]:

# PSEUDOCODE
# 1. Locate the existing evidence-readiness report.
# 2. Read its current contents.
# 3. Compare it with our verified evidence database.
# 4. Identify any outdated information.
# 5. Do not modify any files.

import json
import pandas as pd

report_path = (
    ROOT / "outputs/reports/02_evidence_readiness.json"
)

verified_path = (
    ROOT / "data/processed/02_verified_claims.csv"
)

verified = pd.read_csv(verified_path)

print("Current approved evidence records:", len(verified))

if report_path.exists():
    with open(report_path, "r") as file:
        readiness_report = json.load(file)

    print("\nEXISTING READINESS REPORT")
    print(json.dumps(readiness_report, indent=2))
else:
    print("\nNo existing readiness report found.")

print("\nNo files were modified.")


Current approved evidence records: 3

EXISTING READINESS REPORT
{
  "campaign_id": "CU-001",
  "notebook": "02_Literature_and_Evidence",
  "seed_papers": 8,
  "metadata_online_lookup_enabled": false,
  "metadata_provenance": "previous_metadata_audit_reused_not_requeried",
  "metadata_status_counts": {
    "matching_metadata_needs_manual_review": 8
  },
  "draft_claims": 4,
  "planned_searches_not_executed": 6,
  "human_verified_claims": 3,
  "publications_with_verified_claims": 1,
  "publications_without_verified_claims": 7,
  "verified_claims_eligible_for_retrieval": 3,
  "computational_descriptor_candidates": 6,
  "approved_modeling_descriptors": 0,
  "unresolved_scientific_gaps": 4,
  "evidence_ready_for_rag": false,
  "rag_readiness_note": "Human-verified claims are available for future source-grounded retrieval. The complete RAG pipeline and its retrieval quality have not yet been validated.",
  "ready_for_blinded_benchmark": false,
  "evaluation_status": "planned_not_run",
  "sci


### 2.7b.1 — Evidence-Readiness Report Reconciliation

**Objective:** Reconcile the evidence-readiness report with the current human-reviewed and processed evidence databases.

The original report was generated before completion of the P005 source-review workflow and therefore contains outdated evidence counts.

**Required corrections:**

- Update the human-verified claim count from 0 to 3.
- Record the number of publications represented by verified evidence.
- Record the six extracted computational descriptor candidates.
- Preserve the distinction between evidence retrieval eligibility and modeling readiness.
- Retain the original metadata, literature-search, and evaluation status unless those activities have actually changed.

**Reproducibility requirement:** Human-review decisions and approved evidence records must never be overwritten merely to regenerate a summary report.

**Acceptance criterion:** The updated report must agree with the approved evidence database while preserving all scientifically relevant limitations.


**Report-generation responsibility:** The evidence-readiness report is generated by the canonical, validated export workflow in Section 2.8. No separate report-writing operation is required here.

This subsection documents the reconciliation requirements; the final evidence counts and readiness flags are calculated from the current validated records during the canonical export.



### 2.7b.2 — Safe Notebook Re-execution and Evidence Preservation

**Objective:** Ensure that Notebook 02 can be executed repeatedly without overwriting human-reviewed scientific evidence or introducing inconsistent approval states.

#### Reproducibility requirements

1. Human-review decisions must persist across repeated executions.
2. Approved evidence must be regenerated only from validated human-review records.
3. Repeated execution must not introduce duplicate evidence IDs.
4. Existing verified claims must not be replaced by empty draft datasets.
5. Readiness reports must reflect the current evidence database.
6. Generated files must remain distinguishable from manually reviewed records.

#### Current safeguards

- Three experimental claims from P005 have passed human source review.
- Six published computational descriptor candidates remain unapproved for modeling.
- The evidence-readiness report has been reconciled with the reviewed records.
- A backup of the previous readiness report has been preserved.

**Next action:** Inspect all notebook cells that write evidence files or reports, identify unsafe operations, and correct them before performing a complete notebook rerun.

**Acceptance criterion:** Repeated execution preserves reviewed claims, produces consistent outputs, and passes the project's automated tests.


In [27]:

# PSEUDOCODE
# 1. Open the saved Notebook 02.
# 2. Inspect its code cells.
# 3. Locate references to the approved evidence export and report.
# 4. Display the relevant code for review.
# 5. Do not execute those cells or modify files.

import nbformat

notebook_path = (
    ROOT / "notebooks/02_Literature_and_Evidence.ipynb"
)

notebook = nbformat.read(
    notebook_path,
    as_version=4
)

search_terms = [
    "02_verified_claims.csv",
    "02_evidence_readiness.json",
    "save_csv(",
    "evidence_ready_for_rag"
]

for cell_number, cell in enumerate(notebook.cells, start=1):
    if cell.cell_type != "code":
        continue

    source = cell.source

    if any(term in source for term in search_terms):
        print("\n" + "=" * 60)
        print(f"CODE CELL {cell_number}")
        print("=" * 60)
        print(source[:4500])

print("\nInspection complete. No files modified.")



CODE CELL 21

# PSEUDOCODE
# 1. Read the human-review register.
# 2. Read the processed verified-claims database.
# 3. Check whether our first verified record appears in both.
# 4. Report the result without changing any files.

import pandas as pd

review_path = ROOT / "data/references/02_human_reviewed_claims.csv"
verified_path = ROOT / "data/processed/02_verified_claims.csv"

review_df = pd.read_csv(review_path)
verified_df = pd.read_csv(verified_path)

claim_id = "E-DRAFT-001"

record = review_df.loc[
    review_df["evidence_id"] == claim_id
]

assert len(record) == 1, "Expected exactly one review record."

human_verified = (
    str(record.iloc[0]["source_checked_by_human"]).lower() == "true"
    and record.iloc[0]["review_status"] == "human_verified"
)

in_processed_database = (
    claim_id in verified_df["evidence_id"].values
)

print("Claim ID:", claim_id)
print("Human verification complete:", human_verified)
print("Records in human-review register:", len(review_df))
print("Re

In [28]:

# PSEUDOCODE
# 1. Read the saved Notebook 02.
# 2. Examine only its Python code cells.
# 3. Find commands that may write or overwrite files.
# 4. Print their cell numbers and code lines.
# 5. Do not execute those commands.

import nbformat

notebook_path = (
    ROOT / "notebooks/02_Literature_and_Evidence.ipynb"
)

nb = nbformat.read(notebook_path, as_version=4)

write_patterns = [
    "save_csv(",
    ".to_csv(",
    "json.dump(",
    ".write_text(",
    ".write_bytes(",
    ".to_json(",
    "shutil.copy",
    "shutil.move",
    "os.replace("
]

print("POTENTIAL FILE-WRITING OPERATIONS")
print("=" * 55)

count = 0

for cell_number, cell in enumerate(nb.cells, start=1):

    if cell.cell_type != "code":
        continue

    for line_number, line in enumerate(
        cell.source.splitlines(), start=1
    ):

        if any(p in line for p in write_patterns):
            print(
                f"Cell {cell_number}, "
                f"line {line_number}: {line.strip()[:150]}"
            )
            count += 1

print("=" * 55)
print("Potential write operations found:", count)
print("No files modified.")


POTENTIAL FILE-WRITING OPERATIONS
Cell 24, line 97: reviews.to_csv(review_path, index=False)
Cell 47, line 72: candidates.to_csv(output_path, index=False)
Cell 56, line 23: "save_csv(",
Cell 57, line 18: "save_csv(",
Cell 57, line 19: ".to_csv(",
Cell 57, line 20: "json.dump(",
Cell 57, line 21: ".write_text(",
Cell 57, line 22: ".write_bytes(",
Cell 57, line 23: ".to_json(",
Cell 57, line 24: "shutil.copy",
Cell 57, line 25: "shutil.move",
Cell 57, line 26: "os.replace("
Cell 69, line 29: metadata_path = save_csv(
Cell 69, line 34: queue_path = save_csv(
Cell 69, line 39: search_plan_path = save_csv(
Cell 69, line 44: draft_path = save_csv(
Cell 69, line 139: verified_path = save_csv(
Cell 69, line 280: report_path.write_text(
Potential write operations found: 18
No files modified.


In [29]:

# PSEUDOCODE
# 1. Load the saved Notebook 02.
# 2. Locate the original final-export cell.
# 3. Display its complete source code.
# 4. Do not execute the export code.
# 5. Do not modify any files.

import nbformat

notebook_path = (
    ROOT / "notebooks/02_Literature_and_Evidence.ipynb"
)

nb = nbformat.read(notebook_path, as_version=4)

cell_number = 62

source = nb.cells[cell_number - 1].source

print("ORIGINAL FINAL-EXPORT CELL")
print("=" * 60)

for line_number, line in enumerate(
    source.splitlines(), start=1
):
    print(f"{line_number:03d}: {line}")

print("=" * 60)
print("Inspection only. No files modified.")


ORIGINAL FINAL-EXPORT CELL
001: 
002: ### 2.8.7 — Validation of Human-Reviewed Scientific Claims
003: 
004: **Objective:** Examine the existing evidence-validation rules to ensure that only genuinely human-reviewed, source-traceable scientific claims enter the approved evidence database.
005: 
006: The validation must distinguish between:
007: 
008: - Source metadata verification.
009: - Scientific claim verification.
010: - Computational evidence and mechanistic interpretations.
011: - Human approval for downstream retrieval.
012: 
013: **Validation requirements:**
014: 
015: 1. Every evidence record has a unique identifier.
016: 2. Each approved claim has a valid publication reference.
017: 3. Every approved claim has an identifiable source location.
018: 4. A claim marked `human_verified` must have `source_checked_by_human=True`.
019: 5. Missing or contradictory review information must not silently pass validation.
020: 6. Exporting evidence must not create or alter human-review dec

In [30]:

# PSEUDOCODE
# 1. Read the saved Notebook 02.
# 2. Find the Section 2.8 Markdown heading.
# 3. Locate the first Python code cell after it.
# 4. Display the code without executing it.
# 5. Do not modify any files.

import nbformat

notebook_path = (
    ROOT / "notebooks/02_Literature_and_Evidence.ipynb"
)

nb = nbformat.read(notebook_path, as_version=4)

section_found = False

for index, cell in enumerate(nb.cells):

    if (
        cell.cell_type == "markdown"
        and "2.8 — Human review gate" in cell.source
    ):
        section_found = True
        print("Section 2.8 found at cell:", index + 1)
        continue

    if section_found and cell.cell_type == "code":

        print("\nSECTION 2.8 — EXPORT CODE")
        print("Notebook cell:", index + 1)
        print("=" * 55)

        lines = cell.source.splitlines()

        for number, line in enumerate(lines, start=1):
            print(f"{number:03d}: {line}")

        print("=" * 55)
        break

else:
    print("Export code not located.")

print("\nInspection only. No files modified.")


Export code not located.

Inspection only. No files modified.


In [31]:

# PSEUDOCODE
# 1. Locate the Section 2.8 export code.
# 2. Display its verification and export logic.
# 3. Do not execute the export.
# 4. Do not modify any files.

import nbformat

notebook_path = (
    ROOT / "notebooks/02_Literature_and_Evidence.ipynb"
)

nb = nbformat.read(notebook_path, as_version=4)

# The export cell identified in our latest inspection.
lines = nb.cells[63].source.splitlines()

print("SECTION 2.8 — VERIFICATION AND EXPORT LOGIC")
print("=" * 55)

for number in range(21, min(59, len(lines) + 1)):
    print(f"{number:03d}: {lines[number - 1]}")

print("\nInspection only. No files modified.")


SECTION 2.8 — VERIFICATION AND EXPORT LOGIC
021:     reviews["evidence_id"] == "E-DRAFT-001"
022: ].iloc[0].to_dict()
023: 
024: # Empty FE is intentional for this product-ratio claim.
025: if sample["faradaic_efficiency_percent"] == "":
026:     sample["faradaic_efficiency_percent"] = None
027: 
028: test_cases = {
029:     "Valid human-verified claim": (sample.copy(), True),
030: 
031:     "Verified without human check": (
032:         {**sample, "source_checked_by_human": False},
033:         False
034:     ),
035: 
036:     "Verified without source location": (
037:         {**sample, "exact_source_location": ""},
038:         False
039:     ),
040: 
041:     "Verified without reviewer note": (
042:         {**sample, "reviewer_note": ""},
043:         False
044:     ),
045: 
046:     "Invalid Faradaic efficiency (150%)": (
047:         {
048:             **sample,
049:             "faradaic_efficiency_percent": 150,
050:             "fe_product": "C2H4"
051:         },
052:       


## 2.8 — Human Review Gate and Reproducible Evidence Export

### Objective

Establish a reproducible evidence-validation and export procedure that ensures only scientifically reviewed, source-traceable claims are admitted into CuSelect-AI's approved evidence database.

This stage connects the literature-review workflow to the downstream data-curation, retrieval-augmented generation (RAG), multi-agent reasoning, and catalyst-evaluation components.

### 2.8.1 — Evidence Verification Principles

CuSelect-AI distinguishes between four levels of scientific information:

1. **Bibliographic metadata:** Publication titles, DOIs, authors, and publication dates. Metadata verification does not establish scientific accuracy.
2. **Draft scientific claims:** Extracted or proposed statements requiring comparison with their original sources.
3. **Human-verified evidence:** Claims independently checked against the original publication, with documented source locations and review decisions.
4. **Computational descriptor candidates:** Numerical values extracted from published calculations that require additional assessment of units, reference states, model comparability, and suitability for machine learning.

A scientific claim must never be approved solely because an AI agent generated it, a DOI resolves, or an automated test passes.

### 2.8.2 — Evidence Approval Criteria

A claim may enter the approved evidence database only when:

- The publication and DOI are identifiable.
- The supporting source location is documented.
- The claim accurately represents the source.
- Experimental measurements, computational results, and mechanistic interpretations are distinguished.
- A human reviewer has explicitly completed the source check.
- The required provenance and review fields pass validation.

For an approved claim:

`source_checked_by_human = True`

`review_status = "human_verified"`

These values must reflect an actual human review and must not be assigned automatically by the export workflow.

### 2.8.3 — Current Evidence Status

The initial review of Wang et al. (2024), P005, established:

| Evidence category | Count | Status |
|---|---:|---|
| Seed publications | 8 | Bibliographic register |
| Publications with approved evidence | 1 | P005 |
| Human-verified experimental claims | 3 | Approved |
| Published DFT descriptor candidates | 6 | Source-value checked |
| DFT descriptors approved for modeling | 0 | Not approved |
| Approved mechanistic claims | 0 | Further assessment required |
| Publications without approved claims | 7 | Further review required |

The three verified experimental claims concern ethanol-to-ethylene product ratios and two Faradaic-efficiency observations.

The six computational values were checked against the publisher's original Excel workbook. Their numerical transcription is confirmed, but their suitability for modeling CuSelect-AI's Cu(100) structures has not been established.

### 2.8.4 — Evidence Files and Outputs

The export workflow manages the following files:

**Human-reviewed evidence**

`data/references/02_human_reviewed_claims.csv`

This is the authoritative record of human-review decisions.

**Approved evidence dataset**

`data/processed/02_verified_claims.csv`

Contains only claims satisfying the approved evidence-verification requirements.

**Draft evidence and literature audit**

`data/interim/02_draft_claims.csv`

`data/interim/02_metadata_audit.csv`

`data/interim/02_source_review_queue.csv`

`data/interim/02_search_plan.csv`

These are generated intermediate outputs and must not be confused with approved scientific evidence.

**Computational descriptor candidates**

`data/interim/02_p005_computational_descriptor_candidates.csv`

Contains published numerical DFT values with source provenance and modeling-approval flags.

**Evidence-readiness report**

`outputs/reports/02_evidence_readiness.json`

Summarizes current evidence coverage, verification status, descriptor availability, and downstream readiness.

### 2.8.5 — Reproducibility and Integrity Requirements

The export procedure must:

1. Preserve existing human-review decisions.
2. Reject contradictory or incomplete verification records.
3. Prevent duplicate evidence identifiers.
4. Rebuild approved outputs from the current validated review register.
5. Preserve experimental conditions, units, source locations, and missing values.
6. Avoid automatically approving computational or mechanistic claims.
7. Generate readiness reports from the current validated data.
8. Produce consistent outputs when the same inputs are processed repeatedly.

Human-reviewed source records must never be silently overwritten by generated draft datasets.

### 2.8.6 — Downstream Readiness

**Literature evidence:** Three source-reviewed P005 experimental claims are eligible for future evidence-based retrieval.

**RAG system:** Not yet ready. The complete retrieval workflow, corpus coverage, and evidence-retrieval evaluation remain incomplete.

**Scientific machine learning:** Not yet ready. No published computational descriptor has been approved for direct use with our initial catalyst models.

**Scientific benchmarking:** Not yet performed. Predictive performance, false-advancement rates, citation support, and critic-agent effectiveness remain untested.

### Acceptance Criteria

Section 2.8 is complete when:

- The original export code validates the human-review register correctly.
- All three approved P005 claims remain present after a safe rerun.
- The evidence-readiness report reflects the current database.
- No unverified scientific claims are promoted.
- Automated tests pass.
- Repeated execution cannot silently erase approved evidence.

**Next action:** Inspect and correct the existing export code, validate the approval rules, and test reproducibility before proceeding to Notebook 03 — Data Audit and Curation.



### 2.8.7 — Validation of Human-Reviewed Scientific Claims

**Objective:** Examine the existing evidence-validation rules to ensure that only genuinely human-reviewed, source-traceable scientific claims enter the approved evidence database.

The validation must distinguish between:

- Source metadata verification.
- Scientific claim verification.
- Computational evidence and mechanistic interpretations.
- Human approval for downstream retrieval.

**Validation requirements:**

1. Every evidence record has a unique identifier.
2. Each approved claim has a valid publication reference.
3. Every approved claim has an identifiable source location.
4. A claim marked `human_verified` must have `source_checked_by_human=True`.
5. Missing or contradictory review information must not silently pass validation.
6. Exporting evidence must not create or alter human-review decisions.

**Current approved evidence:** Three experimental observations from P005.

**Procedure:** Inspect the existing `EvidenceClaim` validation model and related verification functions before executing the export workflow.


In [32]:

# PSEUDOCODE
# 1. Import the existing scientific evidence validator.
# 2. Inspect its validation rules.
# 3. Identify how human verification is checked.
# 4. Display the relevant source code.
# 5. Do not modify evidence files.

import inspect

from cuselect.evidence import EvidenceClaim, check_claim_links

print("EVIDENCE CLAIM VALIDATION")
print("=" * 55)

source = inspect.getsource(EvidenceClaim)
print(source)

print("\nPUBLICATION LINK VALIDATION")
print("=" * 55)

link_source = inspect.getsource(check_claim_links)
print(link_source)

print("\nInspection complete.")
print("No evidence files modified.")


EVIDENCE CLAIM VALIDATION
class EvidenceClaim(BaseModel):
    """Single attributable claim. Promotion requires explicit human verification."""
    model_config = ConfigDict(extra="forbid")
    evidence_id: str = Field(min_length=3)
    paper_id: str
    doi: str
    claim_paraphrase: str = Field(min_length=15)
    claim_type: ClaimType
    source_extent: SourceExtent = "unknown"
    exact_source_location: str = ""
    catalyst_system: str = ""
    target_or_observation: str = ""
    conditions: str = ""
    faradaic_efficiency_percent: float | None = Field(default=None, ge=0, le=100)
    fe_product: str = ""
    attribution_note: str = ""
    source_checked_by_human: bool = False
    review_status: ClaimStatus = "pending_source_review"
    reviewer_note: str = ""
    split_group: str = ""

    @model_validator(mode="after")
    def integrity(self) -> "EvidenceClaim":
        self.doi = normalize_doi(self.doi)
        if self.faradaic_efficiency_percent is not None and not self.fe_produ

In [33]:

# PSEUDOCODE
# 1. Load an existing human-verified P005 claim.
# 2. Confirm that the valid claim is accepted.
# 3. Create invalid copies with missing verification details.
# 4. Check whether the validator rejects those copies.
# 5. Report the results without changing files.

import pandas as pd
from cuselect.evidence import EvidenceClaim

review_path = ROOT / "data/references/02_human_reviewed_claims.csv"

reviews = pd.read_csv(
    review_path,
    dtype=str,
    keep_default_na=False
)

sample = reviews.loc[
    reviews["evidence_id"] == "E-DRAFT-001"
].iloc[0].to_dict()

# Empty FE is intentional for this product-ratio claim.
if sample["faradaic_efficiency_percent"] == "":
    sample["faradaic_efficiency_percent"] = None

test_cases = {
    "Valid human-verified claim": (sample.copy(), True),

    "Verified without human check": (
        {**sample, "source_checked_by_human": False},
        False
    ),

    "Verified without source location": (
        {**sample, "exact_source_location": ""},
        False
    ),

    "Verified without reviewer note": (
        {**sample, "reviewer_note": ""},
        False
    ),

    "Invalid Faradaic efficiency (150%)": (
        {
            **sample,
            "faradaic_efficiency_percent": 150,
            "fe_product": "C2H4"
        },
        False
    )
}

results = []

for name, (record, should_pass) in test_cases.items():

    try:
        EvidenceClaim.model_validate(record)
        accepted = True
    except (ValueError, TypeError):
        accepted = False

    results.append({
        "test": name,
        "expected_acceptance": should_pass,
        "actual_acceptance": accepted,
        "result": (
            "PASS" if accepted == should_pass
            else "VALIDATION GAP"
        )
    })

validation_results = pd.DataFrame(results)

display(validation_results)

print(
    "Tests passed:",
    (validation_results["result"] == "PASS").sum(),
    "/",
    len(validation_results)
)

print("No files modified.")


,test,expected_acceptance,actual_acceptance,result
0,Valid human-verified claim,True,True,PASS
1,Verified without human check,False,False,PASS
2,Verified without source location,False,False,PASS
3,Verified without reviewer note,False,False,PASS
4,Invalid Faradaic efficiency (150%),False,False,PASS


Tests passed: 5 / 5
No files modified.



#### 2.8.7.1 — Publication-Link and DOI Integrity

**Objective:** Verify that scientific evidence records are correctly linked to their original publications.

**Required checks:**

1. Each evidence record references a publication in the bibliography.
2. Its DOI agrees with the corresponding bibliographic entry.
3. Evidence identifiers are unique.
4. Invalid or inconsistent publication references are rejected.
5. Publication linkage alone does not establish scientific validity.

**Current status:** The five human-review validation tests passed. Publication-link validation remains to be examined.

**Acceptance criterion:** The evidence-validation system must reject incorrect publication relationships before claims are exported.


In [34]:

# PSEUDOCODE
# 1. Locate the existing publication-link validator.
# 2. Inspect its implementation and function signature.
# 3. Identify which bibliographic relationships it checks.
# 4. Do not execute an evidence export or modify files.

import inspect
from cuselect.evidence import check_claim_links

print("PUBLICATION-LINK VALIDATOR")
print("=" * 55)

print("Function signature:")
print(inspect.signature(check_claim_links))

print("\nSource code:")
print(inspect.getsource(check_claim_links))

print("\nInspection complete.")
print("No files modified.")


PUBLICATION-LINK VALIDATOR
Function signature:
(claims: 'list[EvidenceClaim]', papers: 'pd.DataFrame') -> 'None'

Source code:
def check_claim_links(claims: list[EvidenceClaim], papers: pd.DataFrame) -> None:
    """Raise if a claim is detached from the seed register or duplicates an ID."""
    paper_map = {str(row.paper_id): normalize_doi(row.doi) for row in papers.itertuples(index=False)}
    seen = set()
    for claim in claims:
        if claim.evidence_id in seen:
            raise ValueError(f"Duplicate evidence ID: {claim.evidence_id}")
        seen.add(claim.evidence_id)
        if claim.paper_id not in paper_map or claim.doi != paper_map[claim.paper_id]:
            raise ValueError(f"Claim not linked to the correct paper/DOI: {claim.evidence_id}")


Inspection complete.
No files modified.



#### 2.8.7.2 — Testing Publication-Link Integrity

**Objective:** Test whether the existing publication-link validator correctly identifies inconsistent scientific evidence records.

Four scenarios will be examined:

1. Three valid, human-reviewed P005 evidence records.
2. A duplicated evidence identifier.
3. A claim referencing a nonexistent publication.
4. A claim containing an incorrect DOI.

**Acceptance criterion:** The valid evidence must pass, while all three invalid scenarios must be rejected.

**Scientific limitation:** Correct publication linkage establishes bibliographic consistency, not experimental reproducibility or mechanistic validity.

These tests operate on temporary in-memory records and do not modify the original evidence database.


In [35]:

# PSEUDOCODE
# 1. Load the three approved P005 evidence records.
# 2. Validate their structure using EvidenceClaim.
# 3. Create three deliberately invalid test scenarios.
# 4. Run the publication-link validator on each.
# 5. Compare actual and expected outcomes.
# 6. Do not modify any files.

import pandas as pd
from cuselect.evidence import EvidenceClaim, check_claim_links

review_path = ROOT / "data/references/02_human_reviewed_claims.csv"
papers_path = ROOT / "data/papers.csv"

reviews = pd.read_csv(
    review_path, dtype=str, keep_default_na=False
)

papers_df = pd.read_csv(
    papers_path, dtype=str, keep_default_na=False
)

# Convert existing CSV records to validated objects.
claims = []

for record in reviews.to_dict("records"):

    if record["faradaic_efficiency_percent"] == "":
        record["faradaic_efficiency_percent"] = None

    claims.append(EvidenceClaim.model_validate(record))

assert len(claims) == 3

# Construct test cases in memory only.
test_cases = [
    ("Valid P005 claims", claims, True),

    ("Duplicate evidence ID",
     claims + [claims[0]], False),

    ("Unknown publication",
     [claims[0].model_copy(
         update={"paper_id": "P999"}
     )], False),

    ("Incorrect DOI",
     [claims[0].model_copy(
         update={"doi": "10.0000/incorrect-doi"}
     )], False),
]

results = []

for name, records, expected in test_cases:

    try:
        check_claim_links(records, papers_df)
        accepted = True
    except ValueError:
        accepted = False

    results.append({
        "test": name,
        "expected_acceptance": expected,
        "actual_acceptance": accepted,
        "result": (
            "PASS" if accepted == expected
            else "VALIDATION GAP"
        )
    })

link_test_results = pd.DataFrame(results)

display(link_test_results)

passed = (link_test_results["result"] == "PASS").sum()

print(f"Publication-link tests passed: {passed}/4")
print("No files modified.")


,test,expected_acceptance,actual_acceptance,result
0,Valid P005 claims,True,True,PASS
1,Duplicate evidence ID,False,False,PASS
2,Unknown publication,False,False,PASS
3,Incorrect DOI,False,False,PASS


Publication-link tests passed: 4/4
No files modified.


In [36]:

# PSEUDOCODE
# 1. Save the bibliographic audit and pending review queues.
# 2. Read the authoritative human-review register.
# 3. Validate each claim using the existing Pydantic model.
# 4. Check all publication IDs, DOIs, and duplicate IDs.
# 5. Select only explicitly human-verified claims.
# 6. Prevent accidental loss of previously exported claims.
# 7. Export the approved evidence dataset.
# 8. Count computational descriptor candidates separately.
# 9. Generate an accurate evidence-readiness report.
# 10. Never automatically approve scientific claims.

import json
import pandas as pd

from cuselect.evidence import (
    EvidenceClaim,
    check_claim_links,
    claims_frame,
    save_csv,
)


# --------------------------------------------------
# STEP 1 — Save generated literature-review outputs
# --------------------------------------------------

metadata_path = save_csv(
    metadata_audit,
    ROOT / "data/interim/02_metadata_audit.csv"
)

queue_path = save_csv(
    review_queue,
    ROOT / "data/interim/02_source_review_queue.csv"
)

search_plan_path = save_csv(
    search_plan,
    ROOT / "data/interim/02_search_plan.csv"
)

draft_path = save_csv(
    drafts_df,
    ROOT / "data/interim/02_draft_claims.csv"
)


# --------------------------------------------------
# STEP 2 — Load the authoritative human-review data
# --------------------------------------------------

human_review_path = (
    ROOT / "data/references/02_human_reviewed_claims.csv"
)

if not human_review_path.exists():
    raise FileNotFoundError(
        "Human-review register is missing. "
        "Export stopped to protect verified evidence."
    )

human_table = pd.read_csv(
    human_review_path,
    dtype=str,
    keep_default_na=False
)

reviewed_claims = []

for row in human_table.to_dict("records"):

    # Skip completely empty template rows.
    if not str(row.get("evidence_id", "")).strip():
        continue

    # Missing FE is valid for non-FE observations.
    if row.get("faradaic_efficiency_percent") == "":
        row["faradaic_efficiency_percent"] = None

    # Validate scientific and review metadata.
    claim = EvidenceClaim.model_validate(row)

    reviewed_claims.append(claim)


# --------------------------------------------------
# STEP 3 — Validate publication links and approvals
# --------------------------------------------------

check_claim_links(reviewed_claims, papers)

verified_claims = [
    claim
    for claim in reviewed_claims
    if (
        claim.review_status == "human_verified"
        and claim.source_checked_by_human is True
    )
]

# Preserve the existing evidence schema.
verified_df = claims_frame(verified_claims)

verified_output = (
    ROOT / "data/processed/02_verified_claims.csv"
)

# Protect previously exported claims against
# accidental removal by a rerun.
if verified_output.exists():

    existing_df = pd.read_csv(
        verified_output,
        dtype=str,
        keep_default_na=False
    )

    if "evidence_id" not in existing_df.columns:
        raise ValueError(
            "Existing verified database has no evidence_id column."
        )

    existing_ids = set(existing_df["evidence_id"])

    current_ids = set(verified_df["evidence_id"])

    removed_ids = existing_ids - current_ids

    if removed_ids:
        raise RuntimeError(
            "Export stopped: previously verified claims "
            f"would disappear: {sorted(removed_ids)}. "
            "Review the human-review register before proceeding."
        )

# Export only after all validation checks pass.
verified_path = save_csv(
    verified_df,
    verified_output
)


# --------------------------------------------------
# STEP 4 — Audit computational descriptor candidates
# --------------------------------------------------

descriptor_path = (
    ROOT / "data/interim/"
    "02_p005_computational_descriptor_candidates.csv"
)

descriptor_count = 0
approved_descriptor_count = 0

if descriptor_path.exists():

    descriptor_df = pd.read_csv(
        descriptor_path,
        keep_default_na=False
    )

    descriptor_count = len(descriptor_df)

    if "descriptor_approved" not in descriptor_df.columns:
        raise ValueError(
            "Computational descriptor file lacks "
            "the descriptor_approved column."
        )

    approved_descriptor_count = int(
        descriptor_df["descriptor_approved"]
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
        .sum()
    )


# --------------------------------------------------
# STEP 5 — Calculate publication evidence coverage
# --------------------------------------------------

publications_with_evidence = int(
    verified_df["paper_id"].nunique()
)

publications_without_evidence = int(
    papers["paper_id"].nunique()
    - publications_with_evidence
)


# --------------------------------------------------
# STEP 6 — Generate the evidence-readiness report
# --------------------------------------------------

report = {

    "campaign_id": protocol.campaign_id,

    "notebook": "02_Literature_and_Evidence",

    "seed_papers": int(len(papers)),

    "metadata_online_lookup_enabled":
        RUN_LIVE_CROSSREF_LOOKUP,

    "metadata_provenance": metadata_provenance,

    "metadata_status_counts": {
        str(k): int(v)
        for k, v in
        metadata_audit["metadata_status"].value_counts().items()
    },

    "draft_claims": len(claim_drafts),

    "planned_searches_not_executed": len(search_plan),

    "human_verified_claims": len(verified_claims),

    "publications_with_verified_claims":
        publications_with_evidence,

    "publications_without_verified_claims":
        publications_without_evidence,

    "verified_claims_eligible_for_retrieval":
        len(verified_claims),

    "computational_descriptor_candidates":
        descriptor_count,

    "approved_modeling_descriptors":
        approved_descriptor_count,

    "unresolved_scientific_gaps":
        len(scientific_gaps),

    # Verified evidence exists, but the full RAG
    # system has not been implemented or evaluated.
    "evidence_ready_for_rag": False,

    "rag_readiness_note": (
        "Human-verified claims are available for "
        "future source-grounded retrieval. "
        "The complete RAG pipeline and its retrieval "
        "quality have not yet been validated."
    ),

    "ready_for_blinded_benchmark": False,

    "evaluation_status":
        protocol.evaluation.evaluation_status,

    "scientific_selectivity_predictions_available": False,

    "next_notebook": "03_Data_Audit_and_Curation",
}


# --------------------------------------------------
# STEP 7 — Save the evidence-readiness report
# --------------------------------------------------

report_dir = ROOT / "outputs/reports"

report_dir.mkdir(
    parents=True,
    exist_ok=True
)

report_path = (
    report_dir / "02_evidence_readiness.json"
)

report_path.write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8"
)


# --------------------------------------------------
# STEP 8 — Display the final evidence status
# --------------------------------------------------

print("CUSELECT-AI — NOTEBOOK 02 EVIDENCE EXPORT")
print("=" * 50)

print("Metadata audit:", metadata_path.relative_to(ROOT))
print("Source review queue:", queue_path.relative_to(ROOT))
print("Search plan:", search_plan_path.relative_to(ROOT))
print("Draft claims:", draft_path.relative_to(ROOT))
print("Verified claims:", verified_path.relative_to(ROOT))
print("Readiness report:", report_path.relative_to(ROOT))

print("\nSCIENTIFIC EVIDENCE STATUS")
print("-" * 50)

print("Seed publications:", report["seed_papers"])
print("Human-verified claims:", report["human_verified_claims"])

print(
    "Publications with verified claims:",
    report["publications_with_verified_claims"]
)

print(
    "Computational descriptor candidates:",
    report["computational_descriptor_candidates"]
)

print(
    "Approved modeling descriptors:",
    report["approved_modeling_descriptors"]
)

print(
    "Full RAG readiness:",
    report["evidence_ready_for_rag"]
)

print(
    "Ready for blinded scientific benchmarking:",
    report["ready_for_blinded_benchmark"]
)

print("\nHuman-review decisions were not modified.")

display(
    verified_df[
        [
            "evidence_id",
            "paper_id",
            "claim_type",
            "review_status"
        ]
    ]
)


CUSELECT-AI — NOTEBOOK 02 EVIDENCE EXPORT
Metadata audit: data/interim/02_metadata_audit.csv
Source review queue: data/interim/02_source_review_queue.csv
Search plan: data/interim/02_search_plan.csv
Draft claims: data/interim/02_draft_claims.csv
Verified claims: data/processed/02_verified_claims.csv
Readiness report: outputs/reports/02_evidence_readiness.json

SCIENTIFIC EVIDENCE STATUS
--------------------------------------------------
Seed publications: 8
Human-verified claims: 3
Publications with verified claims: 1
Computational descriptor candidates: 6
Approved modeling descriptors: 0
Full RAG readiness: False
Ready for blinded scientific benchmarking: False

Human-review decisions were not modified.


,evidence_id,paper_id,claim_type,review_status
0,E-DRAFT-001,P005,experimental_observation,human_verified
1,E-P005-FE-C2H4-001,P005,experimental_observation,human_verified
2,E-P005-FE-ETOH-001,P005,experimental_observation,human_verified



### 2.8.8 — Safe Evidence Export and Database Synchronization

**Objective:** Ensure that the processed evidence database accurately reflects the current human-reviewed scientific evidence.

Three experimental observations from P005 have passed human source review and publication-link validation.

The export procedure must:

1. Preserve all existing human-review decisions.
2. Include only records satisfying the `human_verified` criteria.
3. Reject duplicate or inconsistent evidence identifiers.
4. Synchronize corrected scientific metadata between the review register and approved dataset.
5. Avoid automatically approving pending scientific claims.
6. Generate reproducible outputs without losing previously verified evidence.

**Current status:**
- Human-verified claims: 3
- Approved computational modeling descriptors: 0
- Full RAG readiness: False

**Procedure:** First compare the human-review register with the processed database. Then execute the canonical Section 2.8 export workflow after confirming its safety.

**Acceptance criterion:** All approved records must match their validated human-review source records.


In [37]:

# PSEUDOCODE
# 1. Load the human-review register and processed dataset.
# 2. Identify all human-verified claims.
# 3. Compare evidence IDs and source categories.
# 4. Identify metadata differences.
# 5. Do not modify any files.

import pandas as pd

review_path = ROOT / "data/references/02_human_reviewed_claims.csv"
verified_path = ROOT / "data/processed/02_verified_claims.csv"

reviews = pd.read_csv(
    review_path, dtype=str, keep_default_na=False
)

processed = pd.read_csv(
    verified_path, dtype=str, keep_default_na=False
)

approved = reviews.loc[
    reviews["review_status"].eq("human_verified")
    & reviews["source_checked_by_human"]
        .str.strip().str.lower().eq("true")
].copy()

assert approved["evidence_id"].is_unique
assert processed["evidence_id"].is_unique

comparison = approved[
    ["evidence_id", "source_extent"]
].merge(
    processed[["evidence_id", "source_extent"]],
    on="evidence_id",
    how="outer",
    suffixes=("_review", "_processed"),
    indicator=True
)

comparison["source_extent_matches"] = (
    comparison["source_extent_review"]
    == comparison["source_extent_processed"]
)

display(comparison)

print("Human-verified claims:", len(approved))
print("Processed claims:", len(processed))

print(
    "Source-category differences:",
    (~comparison["source_extent_matches"]).sum()
)

print("\nRead-only comparison completed.")
print("No files modified.")


,evidence_id,source_extent_review,source_extent_processed,_merge,source_extent_matches
0,E-DRAFT-001,abstract,abstract,both,True
1,E-P005-FE-C2H4-001,full_text,full_text,both,True
2,E-P005-FE-ETOH-001,full_text,full_text,both,True


Human-verified claims: 3
Processed claims: 3
Source-category differences: 0

Read-only comparison completed.
No files modified.



#### 2.8.8.1 — Complete Evidence Synchronization Audit

**Objective:** Verify that all approved scientific evidence records match their authoritative human-review entries.

The audit compares every field in the approved evidence database against the corresponding human-verified source record.

**Acceptance criteria:**

1. All evidence IDs are unique.
2. Every processed record is human-verified.
3. The approved and processed datasets contain identical evidence IDs.
4. All scientific measurements, experimental conditions, source locations, and review metadata match.
5. No existing evidence file is modified during the audit.

**Scientific limitation:** Database consistency confirms faithful preservation of reviewed records, not independent experimental reproducibility.

**Next action:** Perform a read-only, full-record comparison.


In [38]:

# PSEUDOCODE
# 1. Read the human-review and processed evidence files.
# 2. Select genuinely approved human-review records.
# 3. Compare all columns and every field.
# 4. Report any differences.
# 5. Do not modify any files.

import pandas as pd
from pandas.testing import assert_frame_equal

review_path = ROOT / "data/references/02_human_reviewed_claims.csv"
verified_path = ROOT / "data/processed/02_verified_claims.csv"

reviews = pd.read_csv(
    review_path, dtype=str, keep_default_na=False
)

processed = pd.read_csv(
    verified_path, dtype=str, keep_default_na=False
)

human_checked = (
    reviews["source_checked_by_human"]
    .str.strip().str.lower().eq("true")
)

approved = reviews.loc[
    human_checked
    & reviews["review_status"].eq("human_verified")
].copy()

assert reviews["evidence_id"].is_unique
assert processed["evidence_id"].is_unique

assert set(approved["evidence_id"]) == set(
    processed["evidence_id"]
)

assert set(approved.columns) == set(processed.columns)

columns = sorted(approved.columns)

expected = (
    approved[columns]
    .sort_values("evidence_id")
    .reset_index(drop=True)
)

actual = (
    processed[columns]
    .sort_values("evidence_id")
    .reset_index(drop=True)
)

assert_frame_equal(expected, actual)

print("COMPLETE EVIDENCE SYNCHRONIZATION AUDIT")
print("-" * 45)
print("Approved human-review records:", len(expected))
print("Processed evidence records:", len(actual))
print("Fields compared per record:", len(columns))
print("Full-record consistency: PASSED")
print("No files modified.")


COMPLETE EVIDENCE SYNCHRONIZATION AUDIT
---------------------------------------------
Approved human-review records: 3
Processed evidence records: 3
Fields compared per record: 17
Full-record consistency: PASSED
No files modified.



### 2.8.9 — Final Reproducibility and Re-execution Safeguards

**Objective:** Ensure that Notebook 02 can be executed repeatedly without changing human-review decisions, losing verified evidence, or generating misleading readiness reports.

**Current validated state:**

- Three human-verified experimental claims from P005.
- Complete agreement across all 17 fields in the human-review and processed evidence databases.
- Six published DFT descriptor candidates, none approved for modeling.
- Four successful publication-link validation tests.
- Five successful scientific-claim validation tests.

**Remaining safeguards:**

1. Identify one-time cells that modify human-review approval fields.
2. Separate human approval actions from repeatable data-processing operations.
3. Ensure that the canonical export uses existing verified records without generating new approvals.
4. Keep full RAG readiness set to `False` until retrieval implementation and evaluation are complete.
5. Confirm that repeated execution preserves the scientific evidence and reporting results.

**Acceptance criterion:** A complete notebook rerun must preserve human-verified evidence, regenerate consistent outputs, and pass the project's automated tests.


In [39]:

# PSEUDOCODE
# 1. Read the saved notebook.
# 2. Find assignments involving human verification.
# 3. Locate the RAG-readiness calculation.
# 4. Report their cell and line numbers.
# 5. Do not modify any files.

import nbformat

notebook_path = (
    ROOT / "notebooks/02_Literature_and_Evidence.ipynb"
)

notebook = nbformat.read(
    notebook_path,
    as_version=4
)

print("VERIFICATION AND READINESS CODE AUDIT")
print("=" * 55)

for cell_number, cell in enumerate(notebook.cells, start=1):

    if cell.cell_type != "code":
        continue

    lines = cell.source.splitlines()

    for line_number, line in enumerate(lines, start=1):

        relevant = (
            "source_checked_by_human" in line
            or "review_status" in line
            or "evidence_ready_for_rag" in line
        )

        assignment_like = (
            " = " in line
            or ": " in line
        )

        if relevant and assignment_like:
            print(
                f"Cell {cell_number}, "
                f"line {line_number}: "
                f"{line.strip()[:130]}"
            )

print("=" * 55)
print("Audit complete. No files modified.")


VERIFICATION AND READINESS CODE AUDIT
Cell 7, line 17: required = {'metadata_status','title_similarity','year_consistent','claim_review_status'}
Cell 7, line 28: metadata_cols = ['paper_id','doi','metadata_status','title_similarity','year_consistent','claim_review_status']
Cell 11, line 22: review_queue['source_review_status'] = 'pending_human_review'
Cell 24, line 39: "source_checked_by_human": False,
Cell 24, line 40: "review_status": "pending_source_review",
Cell 24, line 68: "source_checked_by_human": False,
Cell 24, line 69: "review_status": "pending_source_review",
Cell 30, line 19: "review_status": "pending_source_review"
Cell 30, line 28: "review_status": "pending_source_review"
Cell 32, line 39: spectral_evidence["review_status"] = "pending_source_review"
Cell 35, line 19: "review_status": "pending_source_review"
Cell 35, line 27: "review_status": "pending_source_review"
Cell 35, line 35: "review_status": "pending_source_review"
Cell 35, line 43: "review_status": "pending_sour


## 2.9 — Scientific Findings, Limitations, and Handoff to Notebook 03

### 2.9.1 — Research Objective and Accomplishments

Notebook 02 established a reproducible, source-traceable scientific evidence workflow for CuSelect-AI.

The workflow includes:

1. Bibliographic metadata auditing.
2. Literature search planning.
3. Source-specific review queues.
4. Structured scientific claim extraction.
5. Explicit human-source verification.
6. Evidence-schema and publication-link validation.
7. Numerical source-data quality control.
8. Reproducible evidence export and readiness reporting.

The distinction between bibliographic verification, experimental observations, published computational results, and mechanistic interpretations has been preserved throughout the workflow.

### 2.9.2 — Verified Experimental Evidence

The initial review of P005 — Wang et al. (2024), Nature Communications — produced three human-verified experimental evidence records.

**Publication DOI:** https://doi.org/10.1038/s41467-024-54636-w

| Evidence ID | Observation | Verification |
|---|---|---|
| E-DRAFT-001 | Ethanol-to-ethylene product ratios of 0.41 for AgCu NW and 4.26 for Ag₁Cu NW | Human-verified |
| E-P005-FE-C2H4-001 | Maximum ethylene FE of 54.9% on AgCu NW at -1.1 V vs RHE | Human-verified |
| E-P005-FE-ETOH-001 | Maximum ethanol FE of 56.3% on Ag₁Cu NW at -1.0 V vs RHE | Human-verified |

The two FE observations relate to flow-cell experiments using 1 M KOH.

These are published experimental observations, not measurements performed by CuSelect-AI.

All three records are stored in the human-review register and the processed verified-evidence database.

### 2.9.3 — Computational Evidence and Data Quality

The original publisher-supplied source-data workbook was inspected.

Twenty-one worksheets were identified, including numerical source data corresponding to Figures 5a and 5c.

Six published computational values were extracted:

- Three H₂O adsorption-energy values.
- Three reported *CHO free-energy levels.

All six values matched their original Excel cells during numerical quality control.

**Computational evidence status:**

- Extracted descriptor candidates: 6
- Source-value matches: 6
- Descriptors approved for predictive modeling: 0
- DFT calculations performed by CuSelect-AI: 0

The published computational models use Cu(111)-based surfaces, whereas CuSelect-AI's initial structures use Cu(100).

Direct transfer of these quantities between the models is not scientifically justified without further compatibility assessment.

### 2.9.4 — Mechanistic Evidence and Limitations

Spectroscopic features and computational reaction-energy profiles were examined to understand the authors' proposed C–C coupling mechanisms.

The authors associate different Ag configurations with changes in surface intermediates and relative ethylene/ethanol formation.

However:

1. Spectroscopic assignments require interpretation.
2. DFT-derived energies depend on the computational models and approximations.
3. A calculated free-energy change is not equivalent to an activation barrier.
4. The proposed mechanisms have not been independently reproduced by CuSelect-AI.
5. No mechanistic claim has been approved as an experimentally established reaction pathway.

The reviewed mechanisms remain useful for hypothesis generation and future Critic Agent evaluation.

### 2.9.5 — Evidence Coverage and System Readiness

| Indicator | Current status |
|---|---:|
| Seed publications | 8 |
| Publications with human-verified evidence | 1 |
| Human-verified experimental claims | 3 |
| Publications without approved evidence | 7 |
| Published computational descriptor candidates | 6 |
| Descriptors approved for modeling | 0 |
| Full RAG readiness | False |
| Blinded scientific benchmarking completed | No |
| Validated catalyst-selectivity predictions | None |

The seven publications without approved evidence are P001, P002, P003, P004, P006, P007, and P008.

Three verified experimental claims provide a limited foundation for future source-grounded retrieval, but do not establish complete literature coverage or predictive-model readiness.

### 2.9.6 — Reproducibility and Scientific Integrity

The final evidence workflow must preserve the following principles:

- Human-review decisions are never generated automatically.
- Approved records pass schema and publication-link validation.
- Published measurements are not presented as model predictions.
- Experimental conditions and source locations remain traceable.
- Generated datasets do not overwrite the authoritative human-review register.
- Missing numerical data are preserved rather than imputed without justification.
- Literature evidence readiness, RAG readiness, and modeling readiness are reported separately.

The canonical export is managed in Section 2.8.

### 2.9.7 — Handoff to Notebook 03: Data Audit and Curation

**Next notebook:** `03_Data_Audit_and_Curation.ipynb`

**Primary inputs:**

- `data/papers.csv`
- `data/processed/02_verified_claims.csv`
- `data/interim/02_p005_computational_descriptor_candidates.csv`
- `outputs/reports/02_evidence_readiness.json`

**Primary objective:**

Assess the quality, completeness, consistency, and scientific comparability of literature-derived electrochemical and computational data before their use in machine-learning workflows.

**Planned activities:**

1. Define standardized data schemas.
2. Audit missing values and measurement units.
3. Distinguish catalyst structures, experimental systems, and computational models.
4. Standardize product identities and electrochemical conditions.
5. Examine dataset provenance and publication-level grouping.
6. Identify data-quality and model-applicability limitations.
7. Prepare a reproducible foundation for subsequent scientific ML analysis.

### Completion Criteria

Notebook 02 may be considered reproducibly completed when:

1. Its Python cells execute successfully in the prepared project environment.
2. All three human-verified evidence records remain preserved.
3. The canonical export produces consistent scientific records and readiness flags.
4. No human-review decision is automatically created or changed.
5. The automated project tests pass.
6. The completed work is committed to version control.

**Conclusion:** Notebook 02 establishes a validated initial scientific evidence foundation. Broader data collection, model training, RAG evaluation, and agent benchmarking remain future research stages.
